## Download and Cleaning

In [ ]:
import os
from pathlib import Path
import math
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── ChEMBL Client ─────────────────────────────────────────────────────────────
from chembl_webresource_client.new_client import new_client

# ── RDKit ─────────────────────────────────────────────────────────────────────
from rdkit import Chem, RDLogger
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog('rdApp.error')

# ==============================================================================
# 1) CONFIGURATION
# ==============================================================================
BASE_DIR = Path(__file__).resolve().parent
TARGET_BASE_DIR = BASE_DIR

ASSAY_TYPE_TO_PROCESS = "IC50"
USE_CACHE_ONLY = False # Set to True if you only want to process existing raw files
UNIT_TO_M = {"PM": 1e-12, "NM": 1e-9, "UM": 1e-6, "MM": 1e-3, "M": 1.0, "µM": 1e-6, "μM": 1e-6}

CHEMBL_TARGET_IDS = [
    "CHEMBL3105", "CHEMBL1824", "CHEMBL4005", "CHEMBL3130", "CHEMBL3267",
    "CHEMBL3145", "CHEMBL4282", "CHEMBL2842", "CHEMBL3650", "CHEMBL2742",
    "CHEMBL1871", "CHEMBL203", "CHEMBL1957", "CHEMBL4630", "CHEMBL279",
    "CHEMBL267", "CHEMBL4722", "CHEMBL2185", "CHEMBL325", "CHEMBL1865",
]

# ==============================================================================
# 2) CHEMICAL STANDARDIZATION (RDKit Compat)
# ==============================================================================
remover = SaltRemover()
_CLEANUP_PARAMS = getattr(rdMolStandardize, "CleanupParameters", lambda: None)()

# Handling different RDKit versions for Normalizer/Reionizer/FragmentChooser
if hasattr(rdMolStandardize, "Normalizer"):
    _normalizer_obj = rdMolStandardize.Normalizer()
    def _normalize_mol(m): return _normalizer_obj.normalize(m)
elif hasattr(rdMolStandardize, "Normalize"):
    def _normalize_mol(m):
        try:
            return rdMolStandardize.Normalize(m, _CLEANUP_PARAMS)
        except TypeError:
            return rdMolStandardize.Normalize(m)
else:
    def _normalize_mol(m): return m

if hasattr(rdMolStandardize, "Reionizer"):
    _reionizer_obj = rdMolStandardize.Reionizer()
    def _reionize_mol(m): return _reionizer_obj.reionize(m)
elif hasattr(rdMolStandardize, "Reionize"):
    def _reionize_mol(m): return rdMolStandardize.Reionize(m)
else:
    def _reionize_mol(m): return m

if hasattr(rdMolStandardize, "LargestFragmentChooser"):
    _largest_frag = rdMolStandardize.LargestFragmentChooser()
    def _choose_largest_frag(m): return _largest_frag.choose(m)
else:
    def _choose_largest_frag(m):
        frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=False)
        return max(frags, key=lambda x: x.GetNumAtoms()) if frags else m

if hasattr(rdMolStandardize, "TautomerEnumerator"):
    _taut = rdMolStandardize.TautomerEnumerator()
    def _canonicalize_taut(m):
        return _taut.Canonicalize(m) if hasattr(_taut, "Canonicalize") else _taut.canonicalize(m)
else:
    def _canonicalize_taut(m): return m

def standardize_molecule(smiles: str) -> Chem.Mol | None:
    if pd.isna(smiles):
        return None
    try:
        m = Chem.MolFromSmiles(smiles)
        if m is None:
            return None
        m = _choose_largest_frag(m)
        m = _normalize_mol(m)
        m = _reionize_mol(m)
        m = _canonicalize_taut(m)
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None

def mol_to_clean_smiles(m: Chem.Mol) -> str | None:
    try:
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return None

# ==============================================================================
# 3) QUALITY / pActivity / AGGREGATION
# ==============================================================================
def to_pActivity(row) -> float | None:
    pv = row.get("pchembl_value", np.nan)
    if pd.notna(pv):
        try:
            v = float(pv)
            if math.isfinite(v):
                return v
        except Exception:
            pass
    sv = row.get("standard_value", np.nan)
    su = row.get("standard_units", None)
    if pd.notna(sv) and su is not None:
        try:
            su = str(su).upper().strip()
            factor = UNIT_TO_M.get(su, None)
            if factor is None:
                return None
            v_m = float(sv) * factor
            if v_m <= 0:
                return None
            return -math.log10(v_m)
        except Exception:
            return None
    return None

def quality_filter(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["standard_type", "assay_type", "standard_units", "standard_relation"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper().str.strip()
    keep = pd.Series(True, index=df.index)
    keep &= (df.get("standard_type", "IC50") == ASSAY_TYPE_TO_PROCESS)
    if "standard_flag" in df.columns:
        keep &= (df["standard_flag"].fillna(1) == 1)
    if "potential_duplicate" in df.columns:
        keep &= (df["potential_duplicate"].fillna(0) == 0)
    if "data_validity_comment" in df.columns:
        keep &= df["data_validity_comment"].isna()
    if "assay_confidence_score" in df.columns:
        keep &= (df["assay_confidence_score"].fillna(0) >= 8)
    if "assay_type" in df.columns:
        keep &= df["assay_type"].isin(["B"])
    keep &= df["canonical_smiles"].notna()
    keep &= df["standard_value"].notna() | df["pchembl_value"].notna()
    return df[keep]

def annotate_censor(row):
    rel = row.get("standard_relation", "=")
    if pd.isna(rel): rel = "="
    rel = str(rel).strip()
    if rel in (">", ">="): return "right_censored"
    if rel in ("<", "<="): return "left_censored"
    return "exact"

def robust_mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return float(np.median(np.abs(x - med)))

def preprocess_to_pIC50_tables(reports: pd.DataFrame):
    qc = {}
    df = quality_filter(reports)
    qc["n_raw"] = len(reports)
    qc["n_after_quality"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    df["censor"] = df.apply(annotate_censor, axis=1)
    qc["censor_counts"] = df["censor"].value_counts(dropna=False).to_dict()

    df["pActivity"] = df.apply(to_pActivity, axis=1)
    df = df[df["pActivity"].notna()]
    qc["n_with_pActivity"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    # Standardize Molecules
    df["Mol"] = [standardize_molecule(smi) for smi in df["canonical_smiles"].tolist()]
    df = df[df["Mol"].notna()]
    qc["n_after_standardize"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    df["std_canonical_smiles"] = df["Mol"].map(mol_to_clean_smiles)

    # Filter for exact matches
    df_exact = df[df["censor"] == "exact"].copy()
    qc["n_exact"] = len(df_exact)
    if df_exact.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    if "assay_id" not in df_exact.columns:
        df_exact["assay_id"] = "NA"

    # Aggregation per Assay
    per_assay = (df_exact
        .groupby(["molecule_chembl_id", "assay_id", "std_canonical_smiles"], as_index=False)
        .agg(pIC50=("pActivity", "median"),
             n_repl=("pActivity", "size"),
             iqr=("pActivity", lambda x: x.quantile(0.75)-x.quantile(0.25))))

    # Aggregation per Molecule
    per_mol = (per_assay
        .groupby(["molecule_chembl_id", "std_canonical_smiles"], as_index=False)
        .agg(pIC50=("pIC50", "median"),
             n_assays=("assay_id", "nunique"),
             n_total=("n_repl", "sum"),
             iqr_median=("pIC50", robust_mad)))

    qc["n_per_assay"] = len(per_assay)
    qc["n_per_mol"] = len(per_mol)
    return per_assay, per_mol, qc

# ==============================================================================
# 4) IO & FETCHING LOGIC
# ==============================================================================
def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def target_paths(target_id: str):
    base = TARGET_BASE_DIR / target_id / "chembl_data"
    return {
        "raw": base / f"{target_id}_raw_{ASSAY_TYPE_TO_PROCESS}.csv",
        "per_assay": base / f"{target_id}_pIC50_by_assay_{ASSAY_TYPE_TO_PROCESS}.csv",
        "per_mol": base / f"{target_id}_pIC50_{ASSAY_TYPE_TO_PROCESS}.csv",
        "qc": base / f"{target_id}_qc_{ASSAY_TYPE_TO_PROCESS}.json",
    }

def fetch_chembl_activities(target_id: str) -> pd.DataFrame:
    activity_client = new_client.activity
    res = activity_client.filter(target_chembl_id=target_id).filter(standard_type=ASSAY_TYPE_TO_PROCESS)
    return pd.DataFrame.from_dict(res)

def retrive_clean_and_cache(target_id: str):
    paths = target_paths(target_id)

    # If processed file exists, return it (to avoid re-downloading/re-processing)
    if paths["per_mol"].is_file():
        print(f"[{target_id}] Processed file found. Loading from cache.")
        per_mol = pd.read_csv(paths["per_mol"])
        try:
            per_assay = pd.read_csv(paths["per_assay"])
        except Exception:
            per_assay = pd.DataFrame()
        try:
            with open(paths["qc"], "r") as f:
                qc = json.load(f)
        except Exception:
            qc = {"note": "qc missing"}
        return per_assay, per_mol, qc

    # Check for raw file
    if not paths["raw"].is_file():
        if USE_CACHE_ONLY:
            return pd.DataFrame(), pd.DataFrame(), {"error": "cache_only and no raw"}
        print(f"[{target_id}] Downloading ChEMBL activities...")
        try:
            reports = fetch_chembl_activities(target_id)
            save_csv(reports, paths["raw"])
        except Exception as e:
            print(f"[{target_id}] Error downloading: {e}")
            return pd.DataFrame(), pd.DataFrame(), {"error": str(e)}
    else:
        print(f"[{target_id}] Raw file found. Loading...")
        reports = pd.read_csv(paths["raw"], low_memory=False)

    print(f"[{target_id}] Processing and Standardizing...")
    per_assay, per_mol, qc = preprocess_to_pIC50_tables(reports)
    
    # Save processed files
    save_csv(per_assay, paths["per_assay"])
    save_csv(per_mol, paths["per_mol"])
    with open(paths["qc"], "w") as f:
        json.dump(qc, f, indent=2)
        
    print(f"[{target_id}] Done. Molecules: {qc.get('n_per_mol', 0)}")
    return per_assay, per_mol, qc

# ==============================================================================
# 5) MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print(f"Starting Download/Processing for {len(CHEMBL_TARGET_IDS)} targets...")
    
    # Sequential execution (simplest for data download part)
    # If parallel is needed, use ProcessPoolExecutor similar to original script
    results = []
    for tid in tqdm(CHEMBL_TARGET_IDS):
        try:
            _, df_mol, qc_data = retrive_clean_and_cache(tid)
            results.append({
                "Target": tid, 
                "Status": "OK" if not df_mol.empty else "Empty/Fail",
                "Count": len(df_mol) if not df_mol.empty else 0
            })
        except Exception as e:
            results.append({"Target": tid, "Status": f"Error: {e}", "Count": 0})
            
    print("\nSummary:")
    print(pd.DataFrame(results))

## Molecular fingerprints

In [ ]:
import pandas as pd
import numpy as np
import importlib
import urllib.request
from pathlib import Path
from tqdm.auto import tqdm

# ── RDKit & Chemoinformatics ──────────────────────────────────────────────────
from rdkit import Chem, Descriptors
from rdkit.Chem import AllChem, EState
from rdkit.Chem.Crippen import MolLogP, MolMR
from rdkit.Chem.rdMolDescriptors import GetUSRCAT

# ── External Libraries ────────────────────────────────────────────────────────
# Ensure you have installed: pip install mordred mol2vec gensim
from mordred import Calculator, descriptors
from mol2vec.features import MolSentence, mol2alt_sentence
from gensim.models import word2vec

# ==============================================================================
# 1) CONFIGURATION
# ==============================================================================
BASE_DIR = Path(__file__).resolve().parent
TARGET_BASE_DIR = BASE_DIR
MAX_ATOMS_FOR_MORDRED = 120  # Optimization: Skip massive molecules for 3D/Complex desc

# Full list of targets from your original Main.py
CHEMBL_TARGET_IDS = [
    "CHEMBL3105", "CHEMBL1824", "CHEMBL4005", "CHEMBL3130", "CHEMBL3267",
    "CHEMBL3145", "CHEMBL4282", "CHEMBL2842", "CHEMBL3650", "CHEMBL2742",
    "CHEMBL1871", "CHEMBL203", "CHEMBL1957", "CHEMBL4630", "CHEMBL279",
    "CHEMBL267", "CHEMBL4722", "CHEMBL2185", "CHEMBL325", "CHEMBL1865",
]

# ── Setup Mol2Vec ─────────────────────────────────────────────────────────────
MOL2VEC_MODEL_PATH = BASE_DIR / "model_300dim.pkl"
if not MOL2VEC_MODEL_PATH.exists():
    print("Downloading Mol2Vec pre-trained model...")
    url = "https://github.com/samoturk/Mol2Vec/blob/master/examples/models/model_300dim.pkl?raw=true"
    try:
        urllib.request.urlretrieve(url, str(MOL2VEC_MODEL_PATH))
        print("Download complete.")
    except Exception as e:
        print(f"Error downloading Mol2Vec model: {e}")

try:
    mol2vec_model = word2vec.Word2Vec.load(str(MOL2VEC_MODEL_PATH))
except Exception:
    mol2vec_model = None
    print("WARNING: Mol2Vec model could not be loaded. Mol2Vec descriptors will be skipped.")

# ── Setup Mordred ─────────────────────────────────────────────────────────────
# We import specific submodules to allow dynamic 2D vs 3D calculation
_desc_Chi    = importlib.import_module("mordred.chi")
_desc_Kappa  = importlib.import_module("mordred.kappa")
_desc_EState = importlib.import_module("mordred.estate")
_desc_WHIM   = importlib.import_module("mordred.whim")
_desc_RDF    = importlib.import_module("mordred.rdf")
_desc_RMSD   = importlib.import_module("mordred.rmsd")

# 2D Calculator (Fast)
mordred_calc_2d = Calculator([_desc_Chi, _desc_Kappa, _desc_EState], ignore_3D=True)
# 3D Calculator (Slower, requires embedding)
mordred_calc_3d = Calculator([_desc_WHIM, _desc_RDF, _desc_RMSD], ignore_3D=False)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================
def load_target_data(target_id):
    """Loads the cleaned pIC50 CSV generated in step 1."""
    path = TARGET_BASE_DIR / target_id / "chembl_data" / f"{target_id}_pIC50_IC50.csv"
    if not path.exists():
        print(f"  [!] Skipped {target_id}: File not found ({path})")
        return None
    
    df = pd.read_csv(path)
    # Reconstruct RDKit Mol objects
    smi_col = "std_canonical_smiles" if "std_canonical_smiles" in df.columns else "canonical_smiles"
    df["Mol"] = df[smi_col].apply(Chem.MolFromSmiles)
    
    # Drop rows where molecule creation failed
    df = df.dropna(subset=["Mol"]).reset_index(drop=True)
    df["num_atoms"] = df["Mol"].apply(lambda m: m.GetNumAtoms())
    return df

def save_descriptor_csv(df_source, df_desc, target_id, name):
    """Saves the calculated descriptors side-by-side with ID and pIC50."""
    save_dir = TARGET_BASE_DIR / target_id / "chembl_data"
    save_dir.mkdir(parents=True, exist_ok=True)
    path = save_dir / f"{target_id}_pIC50_{name}.csv"
    
    # Metadata columns to keep
    meta = df_source[["molecule_chembl_id", "std_canonical_smiles", "pIC50"]].reset_index(drop=True)
    out = pd.concat([meta, df_desc.reset_index(drop=True)], axis=1)
    
    out.to_csv(path, index=False)
    # print(f"    -> Saved {name}")

def _embed_3d(m):
    """Generates a 3D conformer with a fixed seed."""
    m3 = Chem.AddHs(m)
    params = AllChem.ETKDG()
    params.randomSeed = 0xF00D
    if AllChem.EmbedMolecule(m3, params) == 0:
        return m3
    return None

def check_exists(target_id, name):
    save_dir = TARGET_BASE_DIR / target_id / "chembl_data"
    return (save_dir / f"{target_id}_pIC50_{name}.csv").exists()

# ==============================================================================
# 3) DESCRIPTOR GENERATION LOGIC
# ==============================================================================

def run_rdkit_desc(df, target_id):
    # 1. PhysChem
    if not check_exists(target_id, "RDKit_PhysChem"):
        try:
            names = [n for n, _ in Descriptors._descList]
            # Calculate all available RDKit descriptors
            vals = [[f(m) for _, f in Descriptors._descList] for m in df["Mol"]]
            save_descriptor_csv(df, pd.DataFrame(vals, columns=names), target_id, "RDKit_PhysChem")
        except Exception as e: print(f"    [!] PhysChem Error: {e}")

    # 2. Extended
    if not check_exists(target_id, "RDKit_Extended_Desc"):
        try:
            logp = [MolLogP(m) for m in df["Mol"]]
            mr = [MolMR(m) for m in df["Mol"]]
            es = [float(sum(EState.EStateIndices(m))) for m in df["Mol"]]
            ext = pd.DataFrame({"RDKit_LogP": logp, "RDKit_MR": mr, "RDKit_EState_Sum": es})
            save_descriptor_csv(df, ext, target_id, "RDKit_Extended_Desc")
        except Exception as e: print(f"    [!] Extended Error: {e}")

def run_lingo(df, target_id):
    if check_exists(target_id, "LINGO_Kmers"): return
    try:
        k_lens = [3, 4, 5]
        out = pd.DataFrame(index=df.index)
        smiles = df["std_canonical_smiles"].tolist()
        
        for k in k_lens:
            bags = []
            for s in smiles:
                bag = {}
                for i in range(len(s)-k+1):
                    sub = s[i:i+k]
                    bag[sub] = bag.get(sub, 0) + 1
                bags.append(bag)
            tmp = pd.DataFrame(bags).fillna(0)
            tmp.columns = [f"LINGO_K{k}_{c}" for c in tmp.columns]
            # Keep top 100 features per K to save space
            if not tmp.empty:
                keep = tmp.sum().nlargest(100).index
                out = pd.concat([out, tmp[keep]], axis=1)
        
        save_descriptor_csv(df, out, target_id, "LINGO_Kmers")
    except Exception as e: print(f"    [!] LINGO Error: {e}")

def run_mol2vec(df, target_id):
    if not mol2vec_model or check_exists(target_id, "Mol2Vec_Embeddings"): return
    try:
        sentences = [MolSentence(mol2alt_sentence(m, 1)) for m in df["Mol"]]
        vecs = []
        for s in sentences:
            wv = [mol2vec_model.wv[w] for w in s.sentence if w in mol2vec_model.wv.key_to_index]
            vecs.append(np.mean(wv, axis=0) if wv else np.zeros(300))
        
        df_m2v = pd.DataFrame(np.vstack(vecs), columns=[f"Mol2Vec_{i}" for i in range(300)])
        save_descriptor_csv(df, df_m2v, target_id, "Mol2Vec_Embeddings")
    except Exception as e: print(f"    [!] Mol2Vec Error: {e}")

def run_mordred(df, target_id):
    # Filter: Mordred can hang on very large molecules
    sub = df[df["num_atoms"] <= MAX_ATOMS_FOR_MORDRED].copy()
    if sub.empty: return

    # 1. Mordred 2D
    if not check_exists(target_id, "Mordred_2D_Chi_Kappa_EState"):
        try:
            mols = sub["Mol"].tolist()
            d2 = mordred_calc_2d.pandas(mols, nproc=1).apply(pd.to_numeric, errors='coerce')
            save_descriptor_csv(sub, d2, target_id, "Mordred_2D_Chi_Kappa_EState")
        except Exception as e: print(f"    [!] Mordred 2D Error: {e}")

    # 2. Mordred 3D (Requires Embedding)
    if not check_exists(target_id, "Mordred_3D_4D_WHIM_RDF_RMSD"):
        try:
            mols_3d = []
            valid_idxs = []
            for i, m in enumerate(sub["Mol"]):
                m3 = _embed_3d(m)
                if m3:
                    mols_3d.append(m3)
                    valid_idxs.append(sub.index[i])
            
            if mols_3d:
                d3 = mordred_calc_3d.pandas(mols_3d, nproc=1).apply(pd.to_numeric, errors='coerce')
                save_descriptor_csv(sub.loc[valid_idxs], d3, target_id, "Mordred_3D_4D_WHIM_RDF_RMSD")
        except Exception as e: print(f"    [!] Mordred 3D Error: {e}")

def run_usrcat(df, target_id):
    if check_exists(target_id, "Spectrophore_USRCAT"): return
    try:
        mols_3d = []
        valid_idxs = []
        vals = []
        
        # USRCAT requires 3D coordinates
        for i, m in enumerate(df["Mol"]):
            m3 = _embed_3d(m)
            if m3:
                vals.append(np.array(GetUSRCAT(m3)).flatten())
                valid_idxs.append(df.index[i])
                
        if vals:
            cols = [f"USRCAT_{i}" for i in range(60)]
            out = pd.DataFrame(np.vstack(vals), columns=cols)
            save_descriptor_csv(df.loc[valid_idxs], out, target_id, "Spectrophore_USRCAT")
    except Exception as e: print(f"    [!] USRCAT Error: {e}")

# ==============================================================================
# 4) MAIN EXECUTION LOOP
# ==============================================================================
if __name__ == "__main__":
    print(f"Starting Advanced Descriptor Generation for {len(CHEMBL_TARGET_IDS)} targets...")
    
    for tid in tqdm(CHEMBL_TARGET_IDS, desc="Targets"):
        # 1. Load Data
        df = load_target_data(tid)
        if df is None: 
            continue
            
        print(f"Processing {tid} ({len(df)} mols)...")
        
        # 2. Run Generators
        # Comment out lines if you don't need specific descriptors
        run_rdkit_desc(df, tid)
        run_lingo(df, tid)
        run_mol2vec(df, tid)
        run_mordred(df, tid)
        run_usrcat(df, tid)

    print("\n--- All targets processed ---")

## Feature Selection

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# 1. Custom Correlation Filter
#    "High correlated features... Pearson correlation coefficient > 0.95 
#     were identified and removed"
# ==============================================================================

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Filter features that are highly correlated with each other.
    Keeps the first feature encountered and drops subsequent highly correlated ones.
    """
    def __init__(self, threshold: float = 0.95, random_state: int = None):
        self.threshold = threshold
        self.random_state = random_state
        self.to_drop_ = None

    def fit(self, X, y=None):
        # Conversion to DataFrame handles column tracking better, 
        # but assumes X is numpy array in pipeline.
        if isinstance(X, np.ndarray):
            df = pd.DataFrame(X)
        else:
            df = pd.DataFrame(X).copy()
            
        # Calculate Pearson correlation matrix
        corr_matrix = df.corr().abs()
        
        # Select upper triangle of correlation matrix
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # Identify columns to drop (corr > threshold)
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X):
        if self.to_drop_ is None:
            return X
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.to_drop_)
        # If numpy, we need indices (mapped from fit step)
        return np.delete(X, self.to_drop_, axis=1)

# ==============================================================================
# 2. Pipeline Construction
#    "Zero variance filter... custom correlation filter... 
#     Standardized using Z-score normalization (except for tree-based models)"
# ==============================================================================

def get_preprocessing_pipelines(random_seed=42):
    
    # Common filtering pipeline:
    # 1. VarianceThreshold(0.0): Removes zero variance features.
    # 2. CorrelationFilter: Removes features with corr > 0.95.
    feature_selection_pipe = Pipeline([
        ('variance', VarianceThreshold(0.0)), 
        ('corr', CorrelationFilter(threshold=0.95, random_state=random_seed))
    ])

    scaler = StandardScaler()

    # --- Scenario A: Models Sensitive to Scaling (e.g., Linear, SVM, KNN) ---
    # Includes StandardScaler
    linear_pipeline = Pipeline([
        ('scaler', scaler),             # Z-score Normalization
        ('fs', feature_selection_pipe), # Feature Selection
        ('est', LinearRegression())     # Estimator
    ])

    # --- Scenario B: Tree-Based Models (Invariant to Scaling) ---
    # SKIPS StandardScaler
    tree_pipeline = Pipeline([
        ('fs', feature_selection_pipe),             # Feature Selection Only
        ('est', RandomForestRegressor(n_jobs=1))    # Estimator
    ])
    
    return linear_pipeline, tree_pipeline

## Training and Testing

In [ ]:
import os
import sys
import time
import json
import warnings
import traceback
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple, List, Any

# ─── Parallel Processing & Serialization ──────────────────────────────────────
from joblib import Parallel, delayed, dump, load

# ─── Chemoinformatics (RDKit) ─────────────────────────────────────────────────
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# ─── Scikit-Learn Ecosystem ───────────────────────────────────────────────────
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, SelectFromModel, f_regression
from sklearn.model_selection import GroupKFold, HalvingRandomSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score
from sklearn.compose import TransformedTargetRegressor

# ─── Models ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

# Try importing XGBoost (External dependency)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# Suppress warnings for cleaner logs
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ==============================================================================
# 1. PREPROCESSING HELPERS
# ==============================================================================

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Custom Transformer: Drops features with Pearson correlation > threshold.
    """
    def __init__(self, threshold: float = 0.95):
        self.threshold = threshold
        self.to_drop_ = None

    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            df = pd.DataFrame(X)
        else:
            df = pd.DataFrame(X).copy()
            
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X):
        if self.to_drop_ is None:
            return X
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.to_drop_)
        return np.delete(X, self.to_drop_, axis=1)

def make_scaffold_groups(smiles_list: List[str]) -> List[str]:
    """Generates Murcko Scaffold strings for splitting."""
    scaffolds = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s or '')
            if m:
                scaf = MurckoScaffold.GetScaffoldForMol(m)
                scaffolds.append(Chem.MolToSmiles(scaf, canonical=True))
            else:
                scaffolds.append('NA_SCAFFOLD')
        except:
            scaffolds.append('INVALID_SMILES')
    return scaffolds

def get_holdout_mask(df, holdout_frac=0.15, seed=42):
    """Creates a True/False mask for the External Test Set based on Scaffolds."""
    groups = df['scaffold_group'].values
    unique_groups = np.unique(groups)
    rng = np.random.RandomState(seed)
    rng.shuffle(unique_groups)
    
    mask = np.zeros(len(df), dtype=bool)
    total = len(df)
    count = 0
    min_hold = max(1, int(holdout_frac * total * 0.5))
    
    for g in unique_groups:
        if g in ['NA_SCAFFOLD', 'INVALID_SMILES']: continue
        idx = np.where(groups == g)[0]
        mask[idx] = True
        count += len(idx)
        if count / total >= holdout_frac and count >= min_hold:
            break
            
    # Fallback to random if scaffolds fail
    if mask.sum() < min_hold:
        choice = np.random.RandomState(seed + 999).choice(
            df.index, size=max(min_hold, int(holdout_frac*total)), replace=False
        )
        mask = df.index.isin(choice)
    return mask

# ==============================================================================
# 2. WORKER FUNCTION (Executes 1 Task)
# ==============================================================================

def train_and_evaluate_task(
    config: Dict[str, Any],
    dataset_name: str,
    representation: str,
    model_name: str,
    pipeline: Pipeline,
    param_grid: List[Dict]
) -> Dict[str, Any]:
    
    result_payload = {
        'status': 'fail',
        'dataset': dataset_name,
        'representation': representation,
        'model': model_name,
        'ext_result': None,
        'error_info': None
    }

    try:
        base_dir = Path(config['BASE_DIR'])
        out_dir = Path(config['OUT_DIR'])
        
        # --- 1. Load Data ---
        data_dir = base_dir / dataset_name / 'chembl_data'
        base_csv = data_dir / f"{dataset_name}_pIC50_{config['ASSAY']}.csv"
        rep_csv = data_dir / f"{dataset_name}_pIC50_{representation}.csv"
        
        # Merge ID/Activity with Descriptors
        base_df = pd.read_csv(base_csv, usecols=["molecule_chembl_id", "std_canonical_smiles", "pIC50"]).dropna(subset=['pIC50'])
        rep_df = pd.read_csv(rep_csv)
        df = pd.merge(base_df, rep_df, on=["molecule_chembl_id", "std_canonical_smiles"], how='inner')
        
        if 'pIC50_x' in df.columns: # Handle duplicate columns
            df['pIC50'] = df['pIC50_x']
            df = df.drop(columns=['pIC50_x', 'pIC50_y'], errors='ignore')

        # Clean Features
        exclude_cols = {"molecule_chembl_id", "std_canonical_smiles", "pIC50", 'scaffold_group'}
        feature_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
        if len(feature_cols) < 2: raise ValueError("Not enough feature columns found.")
        
        # Simple Imputation (Inf/NaN -> 0)
        df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        
        # --- 2. Scaffold Split ---
        df['scaffold_group'] = make_scaffold_groups(df['std_canonical_smiles'].tolist())
        holdout_mask = get_holdout_mask(df, config['HOLDOUT_FRAC'], config['RNG_SEED'])
        
        train_df = df.loc[~holdout_mask].reset_index(drop=True)
        test_df  = df.loc[holdout_mask].reset_index(drop=True)
        
        X_train = train_df[feature_cols].values
        y_train = train_df['pIC50'].values
        g_train = train_df['scaffold_group'].values
        
        X_test  = test_df[feature_cols].values
        y_test  = test_df['pIC50'].values
        
        # --- 3. Hyperparameter Tuning & Training ---
        t0 = time.time()
        
        # Setup CV
        cv = GroupKFold(n_splits=config['INNER_FOLDS'])
        splits = list(cv.split(X_train, y_train, g_train))
        
        est = clone(pipeline)
        
        if any(param_grid):
            # Use HalvingRandomSearchCV for speed, or fallback to RandomizedSearchCV
            if config['USE_HALVING_SEARCH']:
                search = HalvingRandomSearchCV(
                    est, param_grid, n_candidates=config['N_ITER_SEARCH'],
                    cv=splits, factor=3, min_resources='exhaust',
                    random_state=config['RNG_SEED'], n_jobs=1, verbose=0, error_score='raise'
                )
            else:
                search = RandomizedSearchCV(
                    est, param_grid, n_iter=config['N_ITER_SEARCH'],
                    cv=splits, random_state=config['RNG_SEED'], n_jobs=1, verbose=0
                )
            search.fit(X_train, y_train)
            best_est = search.best_estimator_
        else:
            # No params to tune (e.g., standard LinearRegression)
            best_est = est.fit(X_train, y_train)
            
        train_time = time.time() - t0
        
        # --- 4. External Evaluation ---
        y_pred = best_est.predict(X_test)
        r2_ext = r2_score(y_test, y_pred)
        
        # --- 5. Save Artifacts ---
        # A. Predictions
        pred_df = test_df[['molecule_chembl_id', 'std_canonical_smiles', 'pIC50']].copy()
        pred_df['predicted_pIC50'] = y_pred
        pred_path = out_dir / f"pred_{dataset_name}_{representation}_{model_name}.csv"
        pred_df.to_csv(pred_path, index=False)
        
        # B. Model
        model_dir = out_dir / 'models'
        model_dir.mkdir(exist_ok=True)
        dump(best_est, model_dir / f"{dataset_name}_{representation}_{model_name}.joblib", compress=3)
        
        result_payload['status'] = 'success'
        result_payload['ext_result'] = {
            'dataset': dataset_name,
            'representation': representation,
            'model': model_name,
            'R2': r2_ext,
            'train_time': train_time
        }
        
    except Exception as e:
        result_payload['error_info'] = {'msg': str(e), 'trace': traceback.format_exc()}
        
    return result_payload

# ==============================================================================
# 3. MAIN RUNNER (Configuration & Model Zoo)
# ==============================================================================

class QSARRunner:
    # --- Configuration ---
    BASE_DIR = Path('../Documents/ChEMBL_data')  # Change to your path
    OUT_DIR = Path('./QSAR_Results')
    ASSAY = 'IC50'
    
    USE_GPU = False
    RNG_SEED = 42
    HOLDOUT_FRAC = 0.15
    OUTER_FOLDS = 5 # (Not used in this simpler Holdout script, but good for CV)
    INNER_FOLDS = 3 # For Hyperparam tuning
    N_ITER_SEARCH = 20 # Number of hyperparam combinations to try
    USE_HALVING_SEARCH = True
    
    # Representations to process
    REPRESENTATIONS = [
        "ECFP4", "ECFP6", "MACCS", "AtomPair", "Torsion",
        "RDKit_PhysChem", "RDKit_Extended_Desc", 
        "Mordred_2D_Chi_Kappa_EState", "Mol2Vec_Embeddings"
    ]

    def __init__(self):
        self.OUT_DIR.mkdir(parents=True, exist_ok=True)
        self.summary_path = self.OUT_DIR / "summary_results.csv"

    def _get_model_zoo(self) -> Dict[str, Tuple[Pipeline, List[Dict]]]:
        """
        DEFINES ALL MODELS AND HYPERPARAMETER GRIDS.
        """
        scaler = StandardScaler()
        # Base Filtering: Remove Zero Variance & High Correlation
        fixed_pipe = Pipeline([
            ('variance', VarianceThreshold(0.0)), 
            ('corr', CorrelationFilter(threshold=0.95))
        ])
        
        # Feature Selection Grid (Lasso vs F_Regression vs None)
        fs_pipe = Pipeline([('fixed', fixed_pipe), ('selector', 'passthrough')])
        
        fs_spaces = [
            {'selector': ['passthrough']},
            {'selector': [SelectPercentile(score_func=f_regression)], 'selector__percentile': [25, 50, 75]},
            {'selector': [SelectFromModel(Lasso(alpha=0.01, random_state=self.RNG_SEED))], 'selector__threshold': ['median', 'mean']}
        ]
        
        def combine(est_grid, fs_grids=fs_spaces):
            """Combines estimator params with feature selection params"""
            return [
                {**e, **{f'fs__{k}': v for k, v in f.items()}} 
                for e in (est_grid if isinstance(est_grid, list) else [est_grid or {}])
                for f in fs_grids
            ]

        M = {}
        
        M['Ridge'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', Ridge(random_state=self.RNG_SEED))]), 
            combine({'est__alpha': np.logspace(-4, 4, 20)})
        )
        M['ElasticNet'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', ElasticNet(random_state=self.RNG_SEED))]), 
            combine({'est__alpha': np.logspace(-4, 2, 10), 'est__l1_ratio': [0.1, 0.5, 0.9]})
        )
        M['BayesianRidge'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', BayesianRidge())]), 
            combine({})
        )
        
        # --- 2. Support Vector Machines (Require Scaling) ---
        M['SVR_rbf'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', SVR(kernel='rbf'))]), 
            combine({'est__C': np.logspace(-2, 3, 5), 'est__gamma': np.logspace(-4, 0, 5)})
        ) 
        # --- 3. Neighbors (Require Scaling) ---
        M['KNN'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', KNeighborsRegressor())]), 
            combine({'est__n_neighbors': [3, 5, 7, 11, 15]})
        )
        
        # --- 4. Tree Ensembles (No Scaling Needed) ---
        M['RandomForest'] = (
            Pipeline([('fs', fs_pipe), ('est', RandomForestRegressor(n_jobs=1, random_state=self.RNG_SEED))]), 
            combine({'est__n_estimators': [100, 300], 'est__max_features': ['sqrt', 'log2']})
        )
        M['ExtraTrees'] = (
            Pipeline([('fs', fs_pipe), ('est', ExtraTreesRegressor(n_jobs=1, random_state=self.RNG_SEED))]), 
            combine({'est__n_estimators': [100, 300], 'est__max_features': ['sqrt', 'log2']})
        )
        M['HistGBDT'] = (
            Pipeline([('fs', fs_pipe), ('est', HistGradientBoostingRegressor(random_state=self.RNG_SEED))]), 
            combine({'est__max_iter': [200, 500], 'est__learning_rate': [0.01, 0.05, 0.1, 0.2]})
        )
        
        if HAS_XGB:
            M['XGBoost'] = (
                Pipeline([('fs', fs_pipe), ('est', XGBRegressor(n_jobs=1, random_state=self.RNG_SEED, verbosity=0))]), 
                combine({'est__n_estimators': [200, 500], 'est__learning_rate': [0.01, 0.05, 0.1], 'est__max_depth': [3, 6, 9]})
            )

        # --- 5. Others ---
        M['PLSRegression'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', PLSRegression())]), 
            combine({'est__n_components': [2, 5, 10]})
        )
        M['MLP'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', MLPRegressor(random_state=self.RNG_SEED, max_iter=500))]), 
            combine({'est__hidden_layer_sizes': [(50,), (100,), (100, 50)], 'est__alpha': [1e-4, 1e-2]})
        )

        return M

    def run(self):
        print(f"--- Starting QSAR Pipeline [Output: {self.OUT_DIR}] ---")
        
        # Discover Datasets
        targets = []
        for p in self.BASE_DIR.glob('CHEMBL*'):
            if (p / 'chembl_data' / f"{p.name}_pIC50_{self.ASSAY}.csv").exists():
                targets.append(p.name)
        
        if not targets:
            print("No datasets found!")
            return

        model_zoo = self._get_model_zoo()
        print(f"Found {len(targets)} targets. Models: {list(model_zoo.keys())}")
        
        # Create Task List
        tasks = []
        for target in targets:
            # Check available descriptors for this target
            avail_reps = []
            for r in self.REPRESENTATIONS:
                if (self.BASE_DIR / target / 'chembl_data' / f"{target}_pIC50_{r}.csv").exists():
                    avail_reps.append(r)
            
            for rep in avail_reps:
                for model_name, (pipe, params) in model_zoo.items():
                    # Check if already done
                    if (self.OUT_DIR / f"pred_{target}_{rep}_{model_name}.csv").exists():
                        continue
                        
                    tasks.append((target, rep, model_name, pipe, params))

        print(f"Total tasks to run: {len(tasks)}")
        if not tasks:
            print("All tasks completed.")
            return

        # Prepare Config Dictionary for Workers
        config = {
            'BASE_DIR': str(self.BASE_DIR),
            'OUT_DIR': str(self.OUT_DIR),
            'ASSAY': self.ASSAY,
            'RNG_SEED': self.RNG_SEED,
            'HOLDOUT_FRAC': self.HOLDOUT_FRAC,
            'INNER_FOLDS': self.INNER_FOLDS,
            'N_ITER_SEARCH': self.N_ITER_SEARCH,
            'USE_HALVING_SEARCH': self.USE_HALVING_SEARCH
        }

        # Run Parallel
        n_cores = max(1, os.cpu_count() - 2)
        print(f"Running on {n_cores} cores...")
        
        results = Parallel(n_jobs=n_cores, verbose=5)(
            delayed(train_and_evaluate_task)(config, *t) for t in tasks
        )
        
        # Process Results
        summary_list = []
        for res in results:
            if res['status'] == 'success':
                summary_list.append(res['ext_result'])
            else:
                print(f"FAIL: {res['dataset']}-{res['model']} -> {res['error_info']['msg']}")
        
        if summary_list:
            df = pd.DataFrame(summary_list)
            header = not self.summary_path.exists()
            df.to_csv(self.summary_path, mode='a', header=header, index=False)
            print(f"Saved summary to {self.summary_path}")

# ==============================================================================
# 4. EXECUTION
# ==============================================================================
if __name__ == '__main__':
    runner = QSARRunner()
    runner.run()

## Statistical Analysis

In [ ]:
from __future__ import annotations
import argparse
import logging
import multiprocessing
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from itertools import combinations
import sys

import matplotlib
matplotlib.use('Agg')  # Must be before importing pyplot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scikit_posthocs as sp
import seaborn as sns
from scipy.stats import (
    friedmanchisquare, kendalltau, pearsonr, spearmanr, 
    wilcoxon, kruskal, shapiro, normaltest
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample
from tqdm import tqdm

# Suppress warnings for cleaner logs
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# ============================================================================== #
# Configuration & Logging
# ============================================================================== #
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

import builtins
_original_open = builtins.open

def _utf8_open(file, mode='r', *args, **kwargs):
    """Wrapper to default to UTF-8 encoding for text files."""
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(file, mode, *args, **kwargs)

builtins.open = _utf8_open

# ============================================================================== #
# 1. Extended QSAR Metrics Calculation
# ============================================================================== #

def calculate_basic_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Calculate basic regression metrics."""
    metrics = {}
    
    # Handle edge cases
    if len(y_true) < 2:
        return {k: np.nan for k in ['R2', 'RMSE', 'MAE', 'Spearman', 'Pearson', 'Kendall_tau', 'CCC']}
    
    # R², RMSE, MAE
    metrics['R2'] = r2_score(y_true, y_pred)
    try:
        metrics['RMSE'] = mean_squared_error(y_true, y_pred, squared=False)
    except TypeError:
        metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
    metrics['MAE'] = mean_absolute_error(y_true, y_pred)
    
    # Correlations
    metrics['Spearman'] = spearmanr(y_true, y_pred).correlation
    metrics['Pearson'] = pearsonr(y_true, y_pred)[0]
    metrics['Kendall_tau'] = kendalltau(y_true, y_pred).correlation
    
    # CCC (Concordance Correlation Coefficient)
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denom = var_true + var_pred + (mean_true - mean_pred)**2
    metrics['CCC'] = (2 * covariance) / denom if denom != 0 else np.nan
    
    return metrics


def calculate_extended_qsar_metrics(y_true: np.ndarray, y_pred: np.ndarray, 
                                     y_train_mean: float = None) -> dict:
    """
    Calculate extended QSAR-specific validation metrics.
    
    References:
    - Tropsha A. (2010) Best Practices for QSAR Model Development
    - Roy K. et al. (2012) rm2 metrics
    - Golbraikh & Tropsha (2002) External validation criteria
    """
    metrics = {}
    n = len(y_true)
    
    if n < 3:
        return {}
    
    if y_train_mean is None:
        y_train_mean = np.mean(y_true)
    
    y_test_mean = np.mean(y_true)
    residuals = y_true - y_pred
    ss_res = np.sum(residuals ** 2)
    ss_tot_test = np.sum((y_true - y_test_mean) ** 2)
    ss_tot_train = np.sum((y_true - y_train_mean) ** 2)
    
    # Q² variants
    metrics['Q2'] = 1 - ss_res / ss_tot_test if ss_tot_test != 0 else np.nan
    metrics['Q2F1'] = 1 - ss_res / ss_tot_train if ss_tot_train != 0 else np.nan
    metrics['Q2F2'] = 1 - ss_res / ss_tot_test if ss_tot_test != 0 else np.nan
    metrics['Q2F3'] = 1 - (ss_res / n) / (ss_tot_train / n) if ss_tot_train != 0 else np.nan
    
    # rm² metrics (Roy et al.)
    r2 = r2_score(y_true, y_pred)
    r = pearsonr(y_true, y_pred)[0]
    
    # Slope through origin: y_true = k * y_pred
    k = np.sum(y_true * y_pred) / np.sum(y_pred ** 2) if np.sum(y_pred ** 2) != 0 else 0
    y_pred_k = k * y_pred
    r2_0 = 1 - np.sum((y_true - y_pred_k) ** 2) / ss_tot_test if ss_tot_test != 0 else np.nan
    
    metrics['rm2'] = r2 * (1 - np.sqrt(np.abs(r2 - r2_0))) if not np.isnan(r2_0) else np.nan
    
    # Reverse rm² (y_pred = k' * y_true)
    k_prime = np.sum(y_true * y_pred) / np.sum(y_true ** 2) if np.sum(y_true ** 2) != 0 else 0
    y_true_k = k_prime * y_true
    ss_tot_pred = np.sum((y_pred - np.mean(y_pred)) ** 2)
    r2_0_prime = 1 - np.sum((y_pred - y_true_k) ** 2) / ss_tot_pred if ss_tot_pred != 0 else np.nan
    
    metrics['rm2_prime'] = r2 * (1 - np.sqrt(np.abs(r2 - r2_0_prime))) if not np.isnan(r2_0_prime) else np.nan
    metrics['rm2_avg'] = (metrics['rm2'] + metrics['rm2_prime']) / 2 if not np.isnan(metrics['rm2']) else np.nan
    metrics['delta_rm2'] = np.abs(metrics['rm2'] - metrics['rm2_prime']) if not np.isnan(metrics['rm2']) else np.nan
    
    # Golbraikh-Tropsha criteria
    metrics['k_slope'] = k
    metrics['k_prime_slope'] = k_prime
    metrics['r2_r0_diff'] = np.abs(r2 - r2_0) if not np.isnan(r2_0) else np.nan
    metrics['r2_r0_prime_diff'] = np.abs(r2 - r2_0_prime) if not np.isnan(r2_0_prime) else np.nan
    
    # Validation criteria checks
    metrics['GT_q2_gt_0.5'] = metrics['Q2F2'] > 0.5 if not np.isnan(metrics['Q2F2']) else False
    metrics['GT_r2_gt_0.6'] = r2 > 0.6
    metrics['GT_k_valid'] = 0.85 <= k <= 1.15
    metrics['GT_k_prime_valid'] = 0.85 <= k_prime <= 1.15
    
    r2_r0_check = min(metrics['r2_r0_diff'], metrics['r2_r0_prime_diff']) < 0.1 if not np.isnan(metrics['r2_r0_diff']) else False
    metrics['GT_r2_r0_valid'] = r2_r0_check
    
    metrics['GT_passes_all'] = all([
        metrics['GT_q2_gt_0.5'],
        metrics['GT_r2_gt_0.6'],
        metrics['GT_r2_r0_valid'],
        metrics['GT_k_valid'] or metrics['GT_k_prime_valid']
    ])
    
    # Roy's criteria
    metrics['Roy_rm2_valid'] = metrics['rm2_avg'] > 0.5 if not np.isnan(metrics['rm2_avg']) else False
    metrics['Roy_delta_valid'] = metrics['delta_rm2'] < 0.2 if not np.isnan(metrics['delta_rm2']) else False
    metrics['Roy_passes'] = metrics['Roy_rm2_valid'] and metrics['Roy_delta_valid']
    
    # Additional error metrics
    metrics['MaxError'] = np.max(np.abs(residuals))
    metrics['MedAE'] = np.median(np.abs(residuals))
    metrics['Bias'] = np.mean(residuals)
    
    # Percentage within thresholds
    metrics['within_0.5_log'] = np.mean(np.abs(residuals) <= 0.5) * 100
    metrics['within_1.0_log'] = np.mean(np.abs(residuals) <= 1.0) * 100
    metrics['within_2.0_log'] = np.mean(np.abs(residuals) <= 2.0) * 100
    
    # MAPE (handle zeros)
    with np.errstate(divide='ignore', invalid='ignore'):
        mape = np.mean(np.abs(residuals / y_true)) * 100
        metrics['MAPE'] = mape if np.isfinite(mape) else np.nan
    
    return metrics


def calculate_all_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Calculate all basic and extended metrics."""
    metrics = calculate_basic_metrics(y_true, y_pred)
    metrics.update(calculate_extended_qsar_metrics(y_true, y_pred))
    return metrics


# ============================================================================== #
# 2. File Processing
# ============================================================================== #

def process_file_wrapper(file_path: Path) -> dict | None:
    """
    Worker function to process a single CSV file.
    Must be top-level for pickle/multiprocessing compatibility.
    """
    try:
        fname = file_path.name
        parts = file_path.stem.split('__')
        
        if len(parts) < 4:
            return None

        # Extract metadata
        dataset = parts[1]
        representation = parts[2]
        model = "__".join(parts[3:])

        # Load Data
        df = pd.read_csv(file_path)
        
        required_cols = ['pIC50', 'pred']
        if not all(col in df.columns for col in required_cols):
            return None
            
        if df.empty or len(df) < 3:
            return None

        y_true = df['pIC50'].values
        y_pred = df['pred'].values

        # Calculate all metrics
        metrics = calculate_all_metrics(y_true, y_pred)
        
        # Add metadata
        metrics.update({
            'dataset': dataset,
            'representation': representation,
            'model': model,
            'source_file': fname,
            'n_samples': len(df),
            'y_true_mean': np.mean(y_true),
            'y_true_std': np.std(y_true),
            'y_true_min': np.min(y_true),
            'y_true_max': np.max(y_true),
            'y_true_range': np.max(y_true) - np.min(y_true)
        })
        
        # Add activity class info if available
        if 'activity_class' in df.columns:
            metrics['has_activity_class'] = True
            metrics['n_activity_classes'] = df['activity_class'].nunique()
        else:
            metrics['has_activity_class'] = False
            
        # Add difficulty score if available
        if 'difficulty_score' in df.columns:
            metrics['mean_difficulty'] = df['difficulty_score'].mean()
            metrics['has_difficulty'] = True
        else:
            metrics['has_difficulty'] = False
            
        return metrics

    except Exception as e:
        return {'error': str(e), 'file': str(file_path)}


def load_predictions_from_file(file_path: Path) -> pd.DataFrame | None:
    """Load prediction data from a CSV file."""
    try:
        df = pd.read_csv(file_path)
        if 'pIC50' in df.columns and 'pred' in df.columns:
            return df
    except Exception:
        pass
    return None


# ============================================================================== #
# 3. Plotting Utilities
# ============================================================================== #

def plot_heatmap(df: pd.DataFrame, output_dir: Path, metric: str = 'R2'):
    """Generate performance heatmap for Dataset x Model."""
    try:
        pivot = df.pivot_table(index='dataset', columns='model', values=metric, aggfunc='max')
        
        # Dynamic figure size
        n_models = len(pivot.columns)
        n_datasets = len(pivot.index)
        fig_width = max(12, n_models * 0.8)
        fig_height = max(8, n_datasets * 0.4)
        
        plt.figure(figsize=(fig_width, fig_height))
        
        # Determine colormap based on metric
        if metric in ['RMSE', 'MAE', 'MaxError', 'MedAE']:
            cmap = 'viridis_r'  # Lower is better
        else:
            cmap = 'viridis'  # Higher is better
        
        sns.heatmap(pivot, annot=True, cmap=cmap, fmt=".2f", linewidths=.5,
                    annot_kws={'size': 8})
        plt.title(f"Dataset x Model Performance Heatmap ({metric})")
        plt.xlabel('Model')
        plt.ylabel('Dataset')
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        plt.savefig(output_dir / f"heatmap_{metric}.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        logger.info(f"Heatmap for {metric} saved.")
    except Exception as e:
        logger.error(f"Error generating heatmap for {metric}: {e}")


def plot_critical_difference(pivot_df: pd.DataFrame, output_dir: Path):
    """Generate Critical Difference diagram using average ranks."""
    try:
        # Calculate average ranks (lower rank = better performance)
        ranks = pivot_df.rank(axis=1, ascending=False)
        avg_ranks = ranks.mean().sort_values()
        
        n_models = len(avg_ranks)
        n_datasets = len(pivot_df)
        
        fig, ax = plt.subplots(figsize=(12, max(4, n_models * 0.35)))
        
        # Plot average ranks
        colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, n_models))
        y_positions = np.arange(n_models)
        
        bars = ax.barh(y_positions, avg_ranks.values, color=colors, alpha=0.8, edgecolor='black')
        ax.set_yticks(y_positions)
        ax.set_yticklabels(avg_ranks.index, fontsize=9)
        ax.set_xlabel('Average Rank (lower is better)', fontsize=10)
        ax.set_title(f'Model Rankings Across {n_datasets} Datasets', fontsize=12)
        
        # Add rank values
        for i, (model, rank) in enumerate(avg_ranks.items()):
            ax.text(rank + 0.1, i, f'{rank:.2f}', va='center', fontsize=9)
        
        ax.set_xlim(0, max(avg_ranks.values) * 1.15)
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_dir / "critical_difference_plot.png", dpi=300)
        plt.close()
        
        # Save ranks
        rank_df = avg_ranks.to_frame('avg_rank')
        rank_df['std_rank'] = ranks.std()
        rank_df.to_csv(output_dir / "model_average_ranks.csv")
        
        logger.info("Critical difference plot saved.")
        
    except Exception as e:
        logger.error(f"Error generating CD plot: {e}")


def plot_metric_comparison(df: pd.DataFrame, output_dir: Path):
    """Plot comparison of multiple metrics across models."""
    metrics = ['R2', 'Q2', 'RMSE', 'MAE', 'Spearman', 'CCC', 'rm2_avg']
    available_metrics = [m for m in metrics if m in df.columns and df[m].notna().sum() > 0]
    
    if len(available_metrics) < 2:
        return
    
    n_metrics = len(available_metrics)
    n_cols = min(3, n_metrics)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_2d(axes).flatten()
    
    for idx, metric in enumerate(available_metrics):
        ax = axes[idx]
        ascending = metric in ['RMSE', 'MAE', 'MaxError', 'MedAE']
        model_stats = df.groupby('model')[metric].mean().sort_values(ascending=ascending)
        
        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(model_stats)))
        if ascending:
            colors = colors[::-1]
        
        ax.barh(range(len(model_stats)), model_stats.values, color=colors)
        ax.set_yticks(range(len(model_stats)))
        ax.set_yticklabels(model_stats.index, fontsize=8)
        ax.set_xlabel(metric)
        ax.set_title(f'Mean {metric}')
        ax.grid(axis='x', alpha=0.3)
    
    # Hide empty subplots
    for idx in range(len(available_metrics), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_dir / "metric_comparison.png", dpi=300)
    plt.close()
    logger.info("Metric comparison plot saved.")


def plot_residual_diagnostics(y_true: np.ndarray, y_pred: np.ndarray, 
                               model_name: str, output_dir: Path):
    """Generate comprehensive residual diagnostic plots."""
    from scipy.stats import probplot
    
    residuals = y_true - y_pred
    std_residuals = (residuals - np.mean(residuals)) / (np.std(residuals) + 1e-10)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Predicted vs Actual
    ax = axes[0, 0]
    ax.scatter(y_true, y_pred, alpha=0.5, s=20, edgecolors='none')
    min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
    ax.set_xlabel('Actual pIC50')
    ax.set_ylabel('Predicted pIC50')
    ax.set_title('Predicted vs Actual')
    ax.legend()
    
    r2 = r2_score(y_true, y_pred)
    ax.annotate(f'R² = {r2:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. Residuals
    # 2. Residuals vs Predicted
    ax = axes[0, 1]
    ax.scatter(y_pred, residuals, alpha=0.5, s=20, edgecolors='none')
    ax.axhline(y=0, color='r', linestyle='--', lw=2)
    ax.axhline(y=np.std(residuals) * 2, color='orange', linestyle=':', alpha=0.7)
    ax.axhline(y=-np.std(residuals) * 2, color='orange', linestyle=':', alpha=0.7)
    ax.set_xlabel('Predicted pIC50')
    ax.set_ylabel('Residuals')
    ax.set_title('Residuals vs Predicted')
    
    # 3. Residuals vs Actual
    ax = axes[0, 2]
    ax.scatter(y_true, residuals, alpha=0.5, s=20, edgecolors='none')
    ax.axhline(y=0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Actual pIC50')
    ax.set_ylabel('Residuals')
    ax.set_title('Residuals vs Actual')
    
    # 4. Residual Distribution
    ax = axes[1, 0]
    ax.hist(residuals, bins=30, edgecolor='black', alpha=0.7, density=True)
    
    # Overlay normal distribution
    x_range = np.linspace(residuals.min(), residuals.max(), 100)
    from scipy.stats import norm
    ax.plot(x_range, norm.pdf(x_range, np.mean(residuals), np.std(residuals)), 
            'r-', lw=2, label='Normal fit')
    ax.axvline(x=0, color='green', linestyle='--', lw=2)
    ax.set_xlabel('Residuals')
    ax.set_ylabel('Density')
    ax.set_title('Residual Distribution')
    ax.legend()
    
    # 5. Q-Q Plot
    ax = axes[1, 1]
    probplot(residuals, dist="norm", plot=ax)
    ax.set_title('Q-Q Plot (Normality Check)')
    
    # 6. Standardized Residuals vs Predicted
    ax = axes[1, 2]
    ax.scatter(y_pred, std_residuals, alpha=0.5, s=20, edgecolors='none')
    ax.axhline(y=0, color='r', linestyle='--', lw=2)
    ax.axhline(y=2, color='orange', linestyle=':', lw=1.5, label='±2 std')
    ax.axhline(y=-2, color='orange', linestyle=':', lw=1.5)
    ax.axhline(y=3, color='red', linestyle=':', lw=1.5, label='±3 std')
    ax.axhline(y=-3, color='red', linestyle=':', lw=1.5)
    ax.set_xlabel('Predicted pIC50')
    ax.set_ylabel('Standardized Residuals')
    ax.set_title('Standardized Residuals')
    ax.legend(loc='upper right')
    
    # Count outliers
    n_outliers_2std = np.sum(np.abs(std_residuals) > 2)
    n_outliers_3std = np.sum(np.abs(std_residuals) > 3)
    ax.annotate(f'Outliers (>2σ): {n_outliers_2std}\nOutliers (>3σ): {n_outliers_3std}', 
                xy=(0.02, 0.98), xycoords='axes fraction', fontsize=9,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(f'Diagnostic Plots: {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / f"diagnostics_{model_name.replace('/', '_').replace(' ', '_')}.png", dpi=300)
    plt.close()


def plot_error_by_activity_range(error_df: pd.DataFrame, output_dir: Path):
    """Plot error analysis by activity range."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. MAE by activity bin (boxplot)
    ax = axes[0, 0]
    if 'activity_bin' in error_df.columns:
        sns.boxplot(data=error_df, x='activity_bin', y='abs_error', ax=ax)
        ax.set_title('Absolute Error by Activity Range')
        ax.set_xlabel('Activity Range')
        ax.set_ylabel('Absolute Error')
        ax.tick_params(axis='x', rotation=45)
    
    # 2. Bias by activity bin (shows systematic over/under prediction)
    ax = axes[0, 1]
    if 'activity_bin' in error_df.columns:
        bias_by_bin = error_df.groupby('activity_bin')['residual'].mean()
        colors = ['red' if x < 0 else 'blue' for x in bias_by_bin.values]
        bars = ax.bar(range(len(bias_by_bin)), bias_by_bin.values, color=colors, alpha=0.7)
        ax.set_xticks(range(len(bias_by_bin)))
        ax.set_xticklabels(bias_by_bin.index, rotation=45)
        ax.axhline(y=0, color='black', linestyle='--', lw=1)
        ax.set_title('Prediction Bias by Activity Range\n(+ve = underprediction, -ve = overprediction)')
        ax.set_xlabel('Activity Range')
        ax.set_ylabel('Mean Residual (Bias)')
    
    # 3. Error distribution by model
    ax = axes[1, 0]
    if 'model' in error_df.columns:
        model_errors = error_df.groupby('model')['abs_error'].mean().sort_values()
        ax.barh(range(len(model_errors)), model_errors.values, color='steelblue', alpha=0.7)
        ax.set_yticks(range(len(model_errors)))
        ax.set_yticklabels(model_errors.index, fontsize=8)
        ax.set_xlabel('Mean Absolute Error')
        ax.set_title('MAE by Model')
        ax.grid(axis='x', alpha=0.3)
    
    # 4. Scatter: Actual vs Error
    ax = axes[1, 1]
    sample = error_df.sample(n=min(5000, len(error_df)), random_state=42)
    scatter = ax.scatter(sample['y_true'], sample['abs_error'], 
                         alpha=0.3, s=10, c=sample['y_true'], cmap='viridis')
    ax.set_xlabel('Actual pIC50')
    ax.set_ylabel('Absolute Error')
    ax.set_title('Error vs Activity Level')
    plt.colorbar(scatter, ax=ax, label='pIC50')
    
    plt.tight_layout()
    plt.savefig(output_dir / "error_by_activity_range.png", dpi=300)
    plt.close()
    logger.info("Error by activity range plot saved.")


def plot_dataset_difficulty(difficulty_df: pd.DataFrame, output_dir: Path):
    """Plot dataset difficulty analysis."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Sort by difficulty
    difficulty_df = difficulty_df.sort_values('difficulty_score', ascending=False)
    
    # 1. Difficulty ranking
    ax = axes[0, 0]
    top_n = min(25, len(difficulty_df))
    top_diff = difficulty_df.head(top_n)
    colors = plt.cm.Reds(np.linspace(0.3, 0.9, top_n))
    ax.barh(range(top_n), top_diff['difficulty_score'].values, color=colors)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_diff['dataset'].values, fontsize=8)
    ax.set_xlabel('Difficulty Score (1 - Best R²)')
    ax.set_title(f'Top {top_n} Most Difficult Datasets')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # 2. Model variance (disagreement)
    ax = axes[0, 1]
    if 'model_variance' in difficulty_df.columns:
        sorted_by_var = difficulty_df.sort_values('model_variance', ascending=False).head(top_n)
        colors = plt.cm.Oranges(np.linspace(0.3, 0.9, top_n))
        ax.barh(range(len(sorted_by_var)), sorted_by_var['model_variance'].values, color=colors)
        ax.set_yticks(range(len(sorted_by_var)))
        ax.set_yticklabels(sorted_by_var['dataset'].values, fontsize=8)
        ax.set_xlabel('Std(R²) Across Models')
        ax.set_title('Datasets with Highest Model Disagreement')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)
    
    # 3. Difficulty vs Model Agreement scatter
    ax = axes[1, 0]
    if 'model_variance' in difficulty_df.columns:
        scatter = ax.scatter(difficulty_df['difficulty_score'], 
                            difficulty_df['model_variance'],
                            c=difficulty_df['best_R2'], cmap='RdYlGn',
                            s=50, alpha=0.7, edgecolors='black', linewidths=0.5)
        ax.set_xlabel('Dataset Difficulty (1 - Best R²)')
        ax.set_ylabel('Model Disagreement (Std R²)')
        ax.set_title('Difficulty vs Model Consensus')
        plt.colorbar(scatter, ax=ax, label='Best R²')
        
        # Annotate extreme points
        for _, row in difficulty_df.nlargest(3, 'difficulty_score').iterrows():
            ax.annotate(row['dataset'][:12], 
                       (row['difficulty_score'], row['model_variance']),
                       fontsize=7, alpha=0.8)
    
    # 4. Distribution of best R² across datasets
    ax = axes[1, 1]
    ax.hist(difficulty_df['best_R2'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(x=0.6, color='red', linestyle='--', lw=2, label='R²=0.6 threshold')
    ax.axvline(x=0.5, color='orange', linestyle='--', lw=2, label='R²=0.5 threshold')
    ax.axvline(x=difficulty_df['best_R2'].median(), color='green', linestyle='-', lw=2, 
               label=f'Median={difficulty_df["best_R2"].median():.2f}')
    ax.set_xlabel('Best R² Achieved')
    ax.set_ylabel('Number of Datasets')
    ax.set_title('Distribution of Best R² Across Datasets')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(output_dir / "dataset_difficulty_analysis.png", dpi=300)
    plt.close()
    logger.info("Dataset difficulty plot saved.")


def plot_model_stability(stability_df: pd.DataFrame, output_dir: Path):
    """Plot model stability analysis."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Stability score ranking
    ax = axes[0, 0]
    sorted_df = stability_df.sort_values('stability_score', ascending=False)
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(sorted_df)))
    ax.barh(range(len(sorted_df)), sorted_df['stability_score'].values, color=colors)
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['model'].values, fontsize=8)
    ax.set_xlabel('Stability Score')
    ax.set_title('Model Stability Ranking\n(Higher = More Consistent)')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # 2. Mean R² vs Std R² scatter
    ax = axes[0, 1]
    scatter = ax.scatter(stability_df['mean_R2'], stability_df['std_R2'],
                        c=stability_df['stability_score'], cmap='RdYlGn',
                        s=100, alpha=0.7, edgecolors='black', linewidths=0.5)
    ax.set_xlabel('Mean R²')
    ax.set_ylabel('Std R² (Variability)')
    ax.set_title('Performance vs Consistency Trade-off')
    plt.colorbar(scatter, ax=ax, label='Stability Score')
    
    # Annotate points
    for _, row in stability_df.iterrows():
        ax.annotate(row['model'][:10], (row['mean_R2'], row['std_R2']),
                   fontsize=7, alpha=0.8)
    
    # 3. CV (Coefficient of Variation) ranking
    ax = axes[1, 0]
    if 'cv_R2' in stability_df.columns:
        sorted_cv = stability_df.sort_values('cv_R2', ascending=True)
        colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_cv)))
        ax.barh(range(len(sorted_cv)), sorted_cv['cv_R2'].values, color=colors)
        ax.set_yticks(range(len(sorted_cv)))
        ax.set_yticklabels(sorted_cv['model'].values, fontsize=8)
        ax.set_xlabel('Coefficient of Variation (CV)')
        ax.set_title('Model Consistency (Lower CV = More Consistent)')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)
    
    # 4. Performance range (min to max)
    ax = axes[1, 1]
    sorted_range = stability_df.sort_values('mean_R2', ascending=False)
    y_pos = np.arange(len(sorted_range))
    
    # Plot range bars
    for i, (_, row) in enumerate(sorted_range.iterrows()):
        ax.plot([row['min_R2'], row['max_R2']], [i, i], 'b-', lw=2, alpha=0.6)
        ax.plot(row['min_R2'], i, 'rv', markersize=8)  # min marker
        ax.plot(row['max_R2'], i, 'g^', markersize=8)  # max marker
        ax.plot(row['mean_R2'], i, 'ko', markersize=6)  # mean marker
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_range['model'].values, fontsize=8)
    ax.set_xlabel('R² Range')
    ax.set_title('Model Performance Range\n(▼=Min, ●=Mean, ▲=Max)')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    ax.axvline(x=0.5, color='orange', linestyle='--', alpha=0.5)
    ax.axvline(x=0.6, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(output_dir / "model_stability_analysis.png", dpi=300)
    plt.close()
    logger.info("Model stability plot saved.")


def plot_representation_analysis(df: pd.DataFrame, output_dir: Path):
    """Analyze and plot representation impact."""
    if 'representation' not in df.columns:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. R² by representation (boxplot)
    ax = axes[0, 0]
    rep_order = df.groupby('representation')['R2'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='representation', y='R2', order=rep_order, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_title('R² Distribution by Representation')
    ax.set_xlabel('Representation')
    ax.set_ylabel('R²')
    
    # 2. Model x Representation heatmap
    ax = axes[0, 1]
    pivot = df.groupby(['model', 'representation'])['R2'].mean().unstack()
    sns.heatmap(pivot, annot=True, cmap='RdYlGn', fmt='.2f', ax=ax,
                annot_kws={'size': 8})
    ax.set_title('Model × Representation Performance')
    ax.set_xlabel('Representation')
    ax.set_ylabel('Model')
    
    # 3. Best representation per model
    ax = axes[1, 0]
    best_rep = df.loc[df.groupby('model')['R2'].idxmax()][['model', 'representation', 'R2']]
    rep_counts = best_rep['representation'].value_counts()
    ax.pie(rep_counts.values, labels=rep_counts.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(rep_counts))))
    ax.set_title('Best Representation Distribution Across Models')
    
    # 4. Summary statistics
    ax = axes[1, 1]
    rep_stats = df.groupby('representation')['R2'].agg(['mean', 'std', 'count'])
    rep_stats = rep_stats.sort_values('mean', ascending=False)
    
    x = np.arange(len(rep_stats))
    bars = ax.bar(x, rep_stats['mean'], yerr=rep_stats['std'], 
                  capsize=5, color='steelblue', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(rep_stats.index, rotation=45, ha='right')
    ax.set_ylabel('Mean R² (± Std)')
    ax.set_title('Representation Performance Summary')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "representation_analysis.png", dpi=300)
    plt.close()
    logger.info("Representation analysis plot saved.")


# ============================================================================== #
# 4. Statistical Methods
# ============================================================================== #

def perform_friedman_test(df: pd.DataFrame, output_dir: Path) -> tuple:
    """Perform Friedman test for model comparison."""
    pivot = df.groupby(['dataset', 'model'])['R2'].max().unstack()
    
    n_datasets_total = pivot.shape[0]
    completeness = pivot.notna().sum()
    valid_models = completeness[completeness == n_datasets_total].index
    
    if len(valid_models) < 2:
        logger.error("Not enough models with complete results for Friedman test.")
        return None, None, None
    
    pivot_clean = pivot[valid_models]
    logger.info(f"Friedman test: {len(valid_models)} models across {n_datasets_total} datasets.")
    
    stat, p_val = friedmanchisquare(*[pivot_clean[c].values for c in pivot_clean.columns])
    
    with open(output_dir / "friedman_stats.txt", "w") as f:
        f.write("=" * 60 + "\n")
        f.write("FRIEDMAN TEST RESULTS\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Number of models compared: {len(valid_models)}\n")
        f.write(f"Number of datasets: {n_datasets_total}\n")
        f.write(f"Friedman Chi-Square statistic: {stat:.4f}\n")
        f.write(f"P-Value: {p_val:.6e}\n")
        f.write(f"Significant at a=0.05: {p_val < 0.05}\n")
        f.write(f"Significant at a=0.01: {p_val < 0.01}\n\n")
        f.write("Models included in analysis:\n")
        for i, m in enumerate(valid_models, 1):
            f.write(f"  {i}. {m}\n")
    
    logger.info(f"Friedman test: χ²={stat:.4f}, p={p_val:.2e}")
    return stat, p_val, pivot_clean


def perform_posthoc_nemenyi(pivot_clean: pd.DataFrame, output_dir: Path):
    """Perform Nemenyi post-hoc test."""
    try:
        nemenyi = sp.posthoc_nemenyi_friedman(pivot_clean)
        nemenyi.to_csv(output_dir / "nemenyi_pvalues.csv")
        
        # Heatmap
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(nemenyi, dtype=bool), k=1)
        sns.heatmap(nemenyi, annot=True, cmap='RdYlGn_r', fmt='.3f',
                    mask=mask, square=True, vmin=0, vmax=1,
                    cbar_kws={'label': 'p-value'}, annot_kws={'size': 8})
        plt.title('Nemenyi Post-hoc Test P-values\n(Green = Significant Difference)')
        plt.tight_layout()
        plt.savefig(output_dir / "nemenyi_heatmap.png", dpi=300)
        plt.close()
        
        # Count significant differences
        sig_count = (nemenyi < 0.05).sum().sum() // 2
        logger.info(f"Nemenyi test: {sig_count} significant pairwise differences (p<0.05)")
        
        return nemenyi
    except Exception as e:
        logger.error(f"Nemenyi test error: {e}")
        return None


def calculate_effect_sizes(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """Calculate Cohen's d effect sizes between all model pairs."""
    models = df['model'].unique()
    results = []
    
    for m1, m2 in combinations(models, 2):
        data1 = df[df['model'] == m1]['R2'].dropna().values
        data2 = df[df['model'] == m2]['R2'].dropna().values
        
        if len(data1) < 2 or len(data2) < 2:
            continue
        
        # Cohen's d
        n1, n2 = len(data1), len(data2)
        var1, var2 = np.var(data1, ddof=1), np.var(data2, ddof=1)
        pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
        
        cohens_d = (np.mean(data1) - np.mean(data2)) / pooled_std if pooled_std > 0 else 0
        
        # Interpretation
        abs_d = np.abs(cohens_d)
        if abs_d < 0.2:
            interpretation = 'negligible'
        elif abs_d < 0.5:
            interpretation = 'small'
        elif abs_d < 0.8:
            interpretation = 'medium'
        else:
            interpretation = 'large'
        
        results.append({
            'model_1': m1,
            'model_2': m2,
            'mean_1': np.mean(data1),
            'mean_2': np.mean(data2),
            'cohens_d': cohens_d,
            'effect_size': interpretation,
            'better_model': m1 if cohens_d > 0 else m2,
            'mean_diff': np.mean(data1) - np.mean(data2)
        })
    
    effect_df = pd.DataFrame(results)
    
    if effect_df.empty:
        return effect_df
    
    effect_df.to_csv(output_dir / "effect_sizes.csv", index=False)
    
    # Create effect size matrix
    effect_matrix = pd.DataFrame(0.0, index=models, columns=models)
    for _, row in effect_df.iterrows():
        effect_matrix.loc[row['model_1'], row['model_2']] = row['cohens_d']
        effect_matrix.loc[row['model_2'], row['model_1']] = -row['cohens_d']
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(effect_matrix.astype(float), annot=True, cmap='RdBu_r',
                center=0, fmt='.2f', square=True, annot_kws={'size': 8})
    plt.title("Cohen's d Effect Sizes Between Models\n(Positive = Row Model Better)")
    plt.tight_layout()
    plt.savefig(output_dir / "effect_size_matrix.png", dpi=300)
    plt.close()
    
    logger.info(f"Effect sizes calculated for {len(results)} model pairs.")
    return effect_df


def perform_pairwise_wilcoxon(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """Perform pairwise Wilcoxon signed-rank tests."""
    models = df['model'].unique()
    results = []
    
    # Create pivot table for paired comparisons
    pivot = df.pivot_table(index='dataset', columns='model', values='R2', aggfunc='max')
    
    for m1, m2 in combinations(models, 2):
        if m1 not in pivot.columns or m2 not in pivot.columns:
            continue
            
        # Get paired data (datasets where both models have results)
        paired_data = pivot[[m1, m2]].dropna()
        
        if len(paired_data) < 5:
            continue
        
        try:
            stat, p_val = wilcoxon(paired_data[m1], paired_data[m2])
            
            results.append({
                'model_1': m1,
                'model_2': m2,
                'n_pairs': len(paired_data),
                'mean_1': paired_data[m1].mean(),
                'mean_2': paired_data[m2].mean(),
                'statistic': stat,
                'p_value': p_val,
                'significant_0.05': p_val < 0.05,
                'significant_0.01': p_val < 0.01,
                'better_model': m1 if paired_data[m1].mean() > paired_data[m2].mean() else m2
            })
        except Exception as e:
            continue
    
    wilcoxon_df = pd.DataFrame(results)
    
    if not wilcoxon_df.empty:
        # Apply Bonferroni correction
        n_tests = len(wilcoxon_df)
        wilcoxon_df['p_value_bonferroni'] = wilcoxon_df['p_value'] * n_tests
        wilcoxon_df['p_value_bonferroni'] = wilcoxon_df['p_value_bonferroni'].clip(upper=1.0)
        wilcoxon_df['significant_bonferroni'] = wilcoxon_df['p_value_bonferroni'] < 0.05
        
        wilcoxon_df = wilcoxon_df.sort_values('p_value')
        wilcoxon_df.to_csv(output_dir / "pairwise_wilcoxon.csv", index=False)
        
        logger.info(f"Wilcoxon tests: {(wilcoxon_df['significant_0.05']).sum()} significant pairs (p<0.05)")
    
    return wilcoxon_df


# ============================================================================== #
# 5. Error Analysis by Activity Class
# ============================================================================== #

def analyze_errors_by_activity_class(output_dir: Path, summary_df: pd.DataFrame) -> pd.DataFrame:
    """Analyze prediction errors stratified by activity class."""
    all_errors = []
    
    # Find all prediction files
    files = list(output_dir.glob("external__*__*__*.csv"))
    
    for file_path in tqdm(files, desc="Analyzing errors by activity class"):
        try:
            data = pd.read_csv(file_path)
            
            if 'pIC50' not in data.columns or 'pred' not in data.columns:
                continue
            
            parts = file_path.stem.split('__')
            if len(parts) < 4:
                continue
                
            dataset = parts[1]
            representation = parts[2]
            model = "__".join(parts[3:])
            
            y_true = data['pIC50'].values
            y_pred = data['pred'].values
            residuals = y_true - y_pred
            
            # Create activity bins if not present
            if 'activity_class' in data.columns:
                activity_bins = data['activity_class']
            else:
                # Create quartile-based bins
                try:
                    activity_bins = pd.qcut(y_true, q=4, 
                                           labels=['Low', 'Med-Low', 'Med-High', 'High'],
                                           duplicates='drop')
                except ValueError:
                    activity_bins = pd.cut(y_true, bins=4, 
                                          labels=['Low', 'Med-Low', 'Med-High', 'High'])
            
            temp_df = pd.DataFrame({
                'dataset': dataset,
                'representation': representation,
                'model': model,
                'y_true': y_true,
                'y_pred': y_pred,
                'residual': residuals,
                'abs_error': np.abs(residuals),
                'squared_error': residuals ** 2,
                'activity_bin': activity_bins
            })
            all_errors.append(temp_df)
            
        except Exception as e:
            continue
    
    if not all_errors:
        logger.warning("No data available for error analysis by activity class.")
        return pd.DataFrame()
    
    error_df = pd.concat(all_errors, ignore_index=True)
    
    # Aggregate statistics by model and activity bin
    agg_stats = error_df.groupby(['model', 'activity_bin']).agg({
        'abs_error': ['mean', 'std', 'median'],
        'residual': 'mean',  # Bias
        'squared_error': 'mean',
        'y_true': 'count'
    }).round(4)
    agg_stats.columns = ['MAE', 'MAE_std', 'MedAE', 'Bias', 'MSE', 'N']
    agg_stats = agg_stats.reset_index()
    agg_stats['RMSE'] = np.sqrt(agg_stats['MSE'])
    
    agg_stats.to_csv(output_dir / "analysis_reports" / "error_by_activity_class.csv", index=False)
    
    # Overall statistics by activity bin
    overall_by_bin = error_df.groupby('activity_bin').agg({
        'abs_error': ['mean', 'std', 'median'],
        'residual': 'mean',
        'y_true': ['count', 'mean', 'std']
    }).round(4)
    overall_by_bin.columns = ['MAE', 'MAE_std', 'MedAE', 'Bias', 'N', 'Mean_Activity', 'Std_Activity']
    overall_by_bin.to_csv(output_dir / "analysis_reports" / "error_by_activity_overall.csv")
    
    # Plot
    plot_error_by_activity_range(error_df, output_dir / "analysis_reports")
    
    # Create detailed heatmaps
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    # MAE heatmap
    mae_pivot = agg_stats.pivot(index='model', columns='activity_bin', values='MAE')
    sns.heatmap(mae_pivot, annot=True, cmap='Reds', fmt='.3f', ax=axes[0], annot_kws={'size': 8})
    axes[0].set_title('MAE by Activity Class')
    
    # Bias heatmap
    bias_pivot = agg_stats.pivot(index='model', columns='activity_bin', values='Bias')
    sns.heatmap(bias_pivot, annot=True, cmap='RdBu_r', center=0, fmt='.3f', ax=axes[1], annot_kws={'size': 8})
    axes[1].set_title('Bias by Activity Class\n(+ve = underprediction)')
    
    # Sample size heatmap
    n_pivot = agg_stats.pivot(index='model', columns='activity_bin', values='N')
    sns.heatmap(n_pivot, annot=True, cmap='Blues', fmt='.0f', ax=axes[2], annot_kws={'size': 8})
    axes[2].set_title('Sample Size by Activity Class')
    
    plt.tight_layout()
    plt.savefig(output_dir / "analysis_reports" / "error_activity_heatmaps.png", dpi=300)
    plt.close()
    
    logger.info(f"Error analysis by activity class completed: {len(error_df)} predictions analyzed.")
    return error_df


# ============================================================================== #
# 6. Dataset Difficulty Analysis
# ============================================================================== #

def analyze_dataset_difficulty(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """Comprehensive dataset difficulty analysis."""
    
    dataset_stats = []
    
    for dataset in df['dataset'].unique():
        subset = df[df['dataset'] == dataset]
        
        r2_values = subset['R2'].dropna()
        
        if len(r2_values) == 0:
            continue
        
        stats = {
            'dataset': dataset,
            'n_models': subset['model'].nunique(),
            'n_representations': subset['representation'].nunique(),
            'n_total_runs': len(subset),
            
            # R² statistics
            'best_R2': r2_values.max(),
            'worst_R2': r2_values.min(),
            'mean_R2': r2_values.mean(),
            'median_R2': r2_values.median(),
            'std_R2': r2_values.std(),
            'iqr_R2': r2_values.quantile(0.75) - r2_values.quantile(0.25),
            
            # Difficulty metrics
            'difficulty_score': 1 - r2_values.max(),
            'model_variance': r2_values.std(),
            'predictability_gap': r2_values.max() - r2_values.min(),
            
            # Best performers
            'best_model': subset.loc[subset['R2'].idxmax(), 'model'] if len(subset) > 0 else None,
            'best_representation': subset.loc[subset['R2'].idxmax(), 'representation'] if len(subset) > 0 else None,
        }
        
        # Additional metrics if available
        if 'RMSE' in subset.columns:
            stats['mean_RMSE'] = subset['RMSE'].mean()
            stats['best_RMSE'] = subset['RMSE'].min()
        
        if 'Q2' in subset.columns:
            stats['best_Q2'] = subset['Q2'].max()
            stats['mean_Q2'] = subset['Q2'].mean()
        
        # Dataset characteristics from summary
        if 'y_true_range' in subset.columns:
            stats['activity_range'] = subset['y_true_range'].iloc[0]
        if 'y_true_std' in subset.columns:
            stats['activity_std'] = subset['y_true_std'].iloc[0]
        if 'n_samples' in subset.columns:
            stats['n_samples'] = subset['n_samples'].iloc[0]
        
        dataset_stats.append(stats)
    
    difficulty_df = pd.DataFrame(dataset_stats)
    
    if difficulty_df.empty:
        logger.warning("No dataset statistics could be computed.")
        return difficulty_df
    
    # Classification of datasets
    difficulty_df['difficulty_class'] = pd.cut(
        difficulty_df['difficulty_score'],
        bins=[-0.01, 0.2, 0.4, 0.6, 1.01],
        labels=['Easy', 'Moderate', 'Hard', 'Very Hard']
    )
    
    # Rank datasets
    difficulty_df['difficulty_rank'] = difficulty_df['difficulty_score'].rank(ascending=False)
    
    # Sort by difficulty
    difficulty_df = difficulty_df.sort_values('difficulty_score', ascending=False)
    difficulty_df.to_csv(output_dir / "dataset_difficulty_analysis.csv", index=False)
    
    # Summary statistics
    summary = {
        'total_datasets': len(difficulty_df),
        'easy_datasets': (difficulty_df['difficulty_class'] == 'Easy').sum(),
        'moderate_datasets': (difficulty_df['difficulty_class'] == 'Moderate').sum(),
        'hard_datasets': (difficulty_df['difficulty_class'] == 'Hard').sum(),
        'very_hard_datasets': (difficulty_df['difficulty_class'] == 'Very Hard').sum(),
        'mean_best_R2': difficulty_df['best_R2'].mean(),
        'median_best_R2': difficulty_df['best_R2'].median(),
        'datasets_above_0.6': (difficulty_df['best_R2'] > 0.6).sum(),
        'datasets_above_0.5': (difficulty_df['best_R2'] > 0.5).sum(),
    }
    
    with open(output_dir / "dataset_difficulty_summary.txt", "w") as f:
        f.write("=" * 60 + "\n")
        f.write("DATASET DIFFICULTY ANALYSIS SUMMARY\n")
        f.write("=" * 60 + "\n\n")
        for k, v in summary.items():
            f.write(f"{k}: {v}\n")
        
        f.write("\n\nTop 10 Most Difficult Datasets:\n")
        f.write("-" * 40 + "\n")
        for _, row in difficulty_df.head(10).iterrows():
            f.write(f"  {row['dataset']}: Best R² = {row['best_R2']:.3f}, "
                   f"Difficulty = {row['difficulty_score']:.3f}\n")
        
        f.write("\n\nTop 10 Easiest Datasets:\n")
        f.write("-" * 40 + "\n")
        for _, row in difficulty_df.tail(10).iterrows():
            f.write(f"  {row['dataset']}: Best R² = {row['best_R2']:.3f}, "
                   f"Difficulty = {row['difficulty_score']:.3f}\n")
    
    # Plot
    plot_dataset_difficulty(difficulty_df, output_dir)
    
    logger.info(f"Dataset difficulty analysis completed: {len(difficulty_df)} datasets analyzed.")
    return difficulty_df


# ============================================================================== #
# 7. Model Stability Analysis
# ============================================================================== #

def analyze_model_stability(df: pd.DataFrame, output_dir: Path, 
                            n_bootstrap: int = 1000) -> pd.DataFrame:
    """Comprehensive model stability and consistency analysis."""
    
    models = df['model'].unique()
    stability_stats = []
    
    for model in models:
        model_data = df[df['model'] == model]
        r2_values = model_data['R2'].dropna().values
        
        if len(r2_values) < 3:
            continue
        
        # Basic statistics
        stats = {
            'model': model,
            'n_datasets': len(r2_values),
            'n_representations': model_data['representation'].nunique(),
            
            # Central tendency
            'mean_R2': np.mean(r2_values),
            'median_R2': np.median(r2_values),
            'trimmed_mean_R2': np.mean(np.sort(r2_values)[1:-1]) if len(r2_values) > 2 else np.mean(r2_values),
            
            # Dispersion
            'std_R2': np.std(r2_values),
            'var_R2': np.var(r2_values),
            'iqr_R2': np.percentile(r2_values, 75) - np.percentile(r2_values, 25),
                        # Range
            'min_R2': np.min(r2_values),
            'max_R2': np.max(r2_values),
            'range_R2': np.max(r2_values) - np.min(r2_values),
            
            # Coefficient of variation (lower = more stable)
            'cv_R2': np.std(r2_values) / np.mean(r2_values) if np.mean(r2_values) != 0 else np.nan,
            
            # Percentiles
            'p10_R2': np.percentile(r2_values, 10),
            'p25_R2': np.percentile(r2_values, 25),
            'p75_R2': np.percentile(r2_values, 75),
            'p90_R2': np.percentile(r2_values, 90),
            
            # Quality counts
            'n_above_0.6': np.sum(r2_values > 0.6),
            'n_above_0.5': np.sum(r2_values > 0.5),
            'n_below_0.3': np.sum(r2_values < 0.3),
            'pct_above_0.6': np.mean(r2_values > 0.6) * 100,
            'pct_above_0.5': np.mean(r2_values > 0.5) * 100,
        }
        
        # Stability score: combines performance and consistency
        # Higher mean, lower std = more stable
        stats['stability_score'] = stats['mean_R2'] * (1 - stats['cv_R2']) if not np.isnan(stats['cv_R2']) else stats['mean_R2']
        
        # Robustness score: based on worst-case performance
        stats['robustness_score'] = stats['p10_R2']  # 10th percentile performance
        
        # Bootstrap confidence intervals
        if n_bootstrap > 0 and len(r2_values) >= 5:
            boot_means = []
            for _ in range(n_bootstrap):
                boot_sample = resample(r2_values, n_samples=len(r2_values))
                boot_means.append(np.mean(boot_sample))
            
            boot_means = np.array(boot_means)
            stats['ci_lower_95'] = np.percentile(boot_means, 2.5)
            stats['ci_upper_95'] = np.percentile(boot_means, 97.5)
            stats['ci_width_95'] = stats['ci_upper_95'] - stats['ci_lower_95']
            stats['bootstrap_std'] = np.std(boot_means)
        
        # Normality test (Shapiro-Wilk)
        if len(r2_values) >= 3 and len(r2_values) <= 5000:
            try:
                _, shapiro_p = shapiro(r2_values)
                stats['shapiro_p'] = shapiro_p
                stats['is_normal'] = shapiro_p > 0.05
            except Exception:
                stats['shapiro_p'] = np.nan
                stats['is_normal'] = None
        
        # Other metrics if available
        for metric in ['RMSE', 'MAE', 'Q2', 'CCC', 'Spearman']:
            if metric in model_data.columns:
                metric_values = model_data[metric].dropna().values
                if len(metric_values) > 0:
                    stats[f'mean_{metric}'] = np.mean(metric_values)
                    stats[f'std_{metric}'] = np.std(metric_values)
        
        stability_stats.append(stats)
    
    stability_df = pd.DataFrame(stability_stats)
    
    if stability_df.empty:
        logger.warning("No model stability statistics could be computed.")
        return stability_df
    
    # Rank models by different criteria
    stability_df['rank_by_mean'] = stability_df['mean_R2'].rank(ascending=False)
    stability_df['rank_by_stability'] = stability_df['stability_score'].rank(ascending=False)
    stability_df['rank_by_robustness'] = stability_df['robustness_score'].rank(ascending=False)
    stability_df['rank_by_consistency'] = stability_df['cv_R2'].rank(ascending=True)  # Lower CV = better
    
    # Combined rank (average of ranks)
    stability_df['combined_rank'] = (
        stability_df['rank_by_mean'] + 
        stability_df['rank_by_stability'] + 
        stability_df['rank_by_robustness']
    ) / 3
    
    # Sort by stability score
    stability_df = stability_df.sort_values('stability_score', ascending=False)
    stability_df.to_csv(output_dir / "model_stability_analysis.csv", index=False)
    
    # Summary report
    with open(output_dir / "model_stability_summary.txt", "w") as f:
        f.write("=" * 60 + "\n")
        f.write("MODEL STABILITY ANALYSIS SUMMARY\n")
        f.write("=" * 60 + "\n\n")
        
        f.write(f"Total models analyzed: {len(stability_df)}\n")
        f.write(f"Bootstrap iterations: {n_bootstrap}\n\n")
        
        f.write("Top 5 Models by Stability Score:\n")
        f.write("-" * 40 + "\n")
        for _, row in stability_df.head(5).iterrows():
            f.write(f"  {row['model'][:30]:<30}: Score={row['stability_score']:.3f}, "
                   f"Mean R²={row['mean_R2']:.3f}, CV={row['cv_R2']:.3f}\n")
        
        f.write("\nTop 5 Models by Mean Performance:\n")
        f.write("-" * 40 + "\n")
        for _, row in stability_df.nlargest(5, 'mean_R2').iterrows():
            f.write(f"  {row['model'][:30]:<30}: Mean R²={row['mean_R2']:.3f} "
                   f"(±{row['std_R2']:.3f})\n")
        
        f.write("\nTop 5 Most Consistent Models (Lowest CV):\n")
        f.write("-" * 40 + "\n")
        for _, row in stability_df.nsmallest(5, 'cv_R2').iterrows():
            f.write(f"  {row['model'][:30]:<30}: CV={row['cv_R2']:.3f}, "
                   f"Mean R²={row['mean_R2']:.3f}\n")
        
        f.write("\nTop 5 Most Robust Models (Best Worst-Case):\n")
        f.write("-" * 40 + "\n")
        for _, row in stability_df.nlargest(5, 'robustness_score').iterrows():
            f.write(f"  {row['model'][:30]:<30}: P10={row['robustness_score']:.3f}, "
                   f"Min={row['min_R2']:.3f}\n")
    
    # Plot
    plot_model_stability(stability_df, output_dir)
    
    logger.info(f"Model stability analysis completed: {len(stability_df)} models analyzed.")
    return stability_df


def analyze_ranking_stability(df: pd.DataFrame, output_dir: Path, 
                               n_bootstrap: int = 1000) -> pd.DataFrame:
    """Analyze stability of model rankings via bootstrap resampling of datasets."""
    
    datasets = df['dataset'].unique()
    models = df['model'].unique()
    
    # Create performance matrix
    perf_matrix = df.pivot_table(index='dataset', columns='model', values='R2', aggfunc='max')
    
    # Only include models with sufficient coverage
    min_coverage = 0.5
    model_coverage = perf_matrix.notna().sum() / len(perf_matrix)
    valid_models = model_coverage[model_coverage >= min_coverage].index.tolist()
    
    if len(valid_models) < 2:
        logger.warning("Not enough models with sufficient coverage for ranking stability analysis.")
        return pd.DataFrame()
    
    perf_matrix = perf_matrix[valid_models]
    
    # Bootstrap rankings
    rank_samples = {m: [] for m in valid_models}
    
    for _ in range(n_bootstrap):
        # Resample datasets with replacement
        sampled_idx = resample(range(len(datasets)), n_samples=len(datasets))
        sampled_datasets = [datasets[i] for i in sampled_idx]
        
        # Get performance for sampled datasets
        sampled_perf = perf_matrix.loc[sampled_datasets].mean()
        
        # Rank models (1 = best)
        ranks = sampled_perf.rank(ascending=False)
        
        for model in valid_models:
            if model in ranks.index:
                rank_samples[model].append(ranks[model])
    
    # Calculate rank statistics
    rank_stats = []
    for model, ranks in rank_samples.items():
        if ranks:
            rank_stats.append({
                'model': model,
                'mean_rank': np.mean(ranks),
                'std_rank': np.std(ranks),
                'median_rank': np.median(ranks),
                'min_rank': np.min(ranks),
                'max_rank': np.max(ranks),
                'ci_lower': np.percentile(ranks, 2.5),
                'ci_upper': np.percentile(ranks, 97.5),
                'prob_top_1': np.mean(np.array(ranks) == 1) * 100,
                'prob_top_3': np.mean(np.array(ranks) <= 3) * 100,
                'prob_top_5': np.mean(np.array(ranks) <= 5) * 100,
                'prob_bottom_3': np.mean(np.array(ranks) > len(valid_models) - 3) * 100,
            })
    
    rank_df = pd.DataFrame(rank_stats).sort_values('mean_rank')
    rank_df.to_csv(output_dir / "ranking_stability.csv", index=False)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 1. Rank distribution boxplot
    ax = axes[0]
    rank_data_for_plot = []
    for model in rank_df['model']:
        for rank in rank_samples[model]:
            rank_data_for_plot.append({'Model': model, 'Rank': rank})
    
    rank_plot_df = pd.DataFrame(rank_data_for_plot)
    order = rank_df['model'].tolist()
    
    sns.boxplot(data=rank_plot_df, x='Model', y='Rank', order=order, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Bootstrap Rank Distribution (n={n_bootstrap})')
    ax.set_ylabel('Rank (lower = better)')
    
    # 2. Mean rank with CI
    ax = axes[1]
    y_pos = np.arange(len(rank_df))
    
    # Calculate error bars (ensure non-negative values)
    err_lower = np.maximum(rank_df['mean_rank'] - rank_df['ci_lower'], 0)
    err_upper = np.maximum(rank_df['ci_upper'] - rank_df['mean_rank'], 0)
    
    ax.errorbar(rank_df['mean_rank'], y_pos,
                xerr=[err_lower, err_upper],
                fmt='o', capsize=4, capthick=1.5, markersize=6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(rank_df['model'], fontsize=8)
    ax.set_xlabel('Mean Rank (95% CI)')
    ax.set_title('Model Ranking Stability')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "ranking_stability.png", dpi=300)
    plt.close()
    
    logger.info(f"Ranking stability analysis completed: {len(rank_df)} models analyzed.")
    return rank_df


# ============================================================================== #
# 8. Extended QSAR Validation Report
# ============================================================================== #

def generate_qsar_validation_report(df: pd.DataFrame, output_dir: Path):
    """Generate comprehensive QSAR validation report based on established criteria."""
    
    report_lines = []
    report_lines.append("=" * 70)
    report_lines.append("QSAR MODEL VALIDATION REPORT")
    report_lines.append("Based on Tropsha, Roy, and Golbraikh-Tropsha Criteria")
    report_lines.append("=" * 70)
    report_lines.append("")
    
    # Check which validation columns are available
    gt_cols = [c for c in df.columns if c.startswith('GT_')]
    roy_cols = [c for c in df.columns if c.startswith('Roy_')]
    
    if gt_cols:
        report_lines.append("\n" + "-" * 50)
        report_lines.append("GOLBRAIKH-TROPSHA CRITERIA SUMMARY")
        report_lines.append("-" * 50)
        
        for model in df['model'].unique():
            model_data = df[df['model'] == model]
            
            if 'GT_passes_all' in model_data.columns:
                passes = model_data['GT_passes_all'].sum()
                total = len(model_data)
                pct = passes / total * 100 if total > 0 else 0
                report_lines.append(f"\n{model}:")
                report_lines.append(f"  Passes all GT criteria: {passes}/{total} datasets ({pct:.1f}%)")
                
                # Individual criteria
                for col in ['GT_q2_gt_0.5', 'GT_r2_gt_0.6', 'GT_k_valid', 'GT_k_prime_valid', 'GT_r2_r0_valid']:
                    if col in model_data.columns:
                        passes_crit = model_data[col].sum()
                        report_lines.append(f"    {col}: {passes_crit}/{total}")
    
    if roy_cols:
        report_lines.append("\n" + "-" * 50)
        report_lines.append("ROY'S rm² CRITERIA SUMMARY")
        report_lines.append("-" * 50)
        
        for model in df['model'].unique():
            model_data = df[df['model'] == model]
            
            if 'Roy_passes' in model_data.columns:
                passes = model_data['Roy_passes'].sum()
                total = len(model_data)
                pct = passes / total * 100 if total > 0 else 0
                report_lines.append(f"\n{model}:")
                report_lines.append(f"  Passes Roy criteria: {passes}/{total} datasets ({pct:.1f}%)")
                
                if 'rm2_avg' in model_data.columns:
                    report_lines.append(f"  Mean rm²_avg: {model_data['rm2_avg'].mean():.3f}")
                if 'delta_rm2' in model_data.columns:
                    report_lines.append(f"  Mean delta_rm²: {model_data['delta_rm2'].mean():.3f}")
    
    # Performance thresholds summary
    report_lines.append("\n" + "-" * 50)
    report_lines.append("PERFORMANCE THRESHOLD SUMMARY")
    report_lines.append("-" * 50)
    
    thresholds = [(0.7, 'R² > 0.7 (Good)'), (0.6, 'R² > 0.6 (Acceptable)'), 
                  (0.5, 'R² > 0.5 (Marginal)')]
    
    for thresh, label in thresholds:
        report_lines.append(f"\n{label}:")
        for model in df['model'].unique():
            model_r2 = df[df['model'] == model]['R2']
            pct_above = (model_r2 > thresh).mean() * 100
            report_lines.append(f"  {model}: {pct_above:.1f}% of datasets")
    
    # Within log unit analysis
    report_lines.append("\n" + "-" * 50)
    report_lines.append("PREDICTION ACCURACY (Within X Log Units)")
    report_lines.append("-" * 50)
    
    for col, label in [('within_0.5_log', '±0.5 log'), ('within_1.0_log', '±1.0 log')]:
        if col in df.columns:
            report_lines.append(f"\nPredictions {label}:")
            for model in df['model'].unique():
                model_data = df[df['model'] == model][col]
                report_lines.append(f"  {model}: {model_data.mean():.1f}%")
    
    # Save report
    report_text = "\n".join(report_lines)
    with open(output_dir / "qsar_validation_report.txt", "w") as f:
        f.write(report_text)
    
    logger.info("QSAR validation report generated.")
    return report_text


# ============================================================================== #
# 9. Main QSARAnalyzer Class
# ============================================================================== #

class QSARAnalyzer:
    """Comprehensive QSAR benchmark analyzer."""
    
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.report_dir = output_dir / "analysis_reports"
        self.report_dir.mkdir(parents=True, exist_ok=True)
        self.summary_path = output_dir / "summary_results_external.csv"
        self.df = None
    
    def build_summary(self) -> bool:
        """Scan directory in parallel and build summary CSV with all metrics."""
        logger.info(f"Scanning {self.output_dir}...")
        
        files = list(self.output_dir.glob("external__*__*__*.csv"))
        
        if not files:
            logger.warning("No prediction files found matching pattern 'external__*__*__*.csv'")
            return False
        
        logger.info(f"Processing {len(files)} files using {multiprocessing.cpu_count()} cores...")
        
        results = []
        errors = []
        
        # Parallel execution (ThreadPoolExecutor is safer in Windows/Jupyter notebooks)
        n_workers = max(1, min(32, multiprocessing.cpu_count() or 1))
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {executor.submit(process_file_wrapper, f): f for f in files}
            for future in tqdm(as_completed(futures), total=len(files), desc="Processing files"):
                try:
                    res = future.result()
                except Exception as e:
                    errors.append({'error': str(e), 'file': str(futures.get(future)) if future in futures else ''})
                    continue
                if res:
                    if 'error' in res:
                        errors.append(res)
                    else:
                        results.append(res)
        
        if errors:
            logger.warning(f"Encountered {len(errors)} errors during processing.")
            error_df = pd.DataFrame(errors)
            error_df.to_csv(self.report_dir / "processing_errors.csv", index=False)
        
        if not results:
            logger.warning("No valid results extracted.")
            return False
        
        self.df = pd.DataFrame(results)
        self.df.to_csv(self.summary_path, index=False)
        logger.info(f"Summary saved: {self.summary_path} ({len(self.df)} rows)")
        
        return True
    
    def load_summary(self) -> bool:
        """Load existing summary file."""
        if self.summary_path.exists():
            self.df = pd.read_csv(self.summary_path)
            logger.info(f"Loaded summary: {len(self.df)} rows")
            return True
        return False
    
    def analyze(self, run_all: bool = True):
        """Run comprehensive analysis pipeline."""
        if self.df is None:
            if not self.load_summary():
                if not self.build_summary():
                    logger.error("Cannot proceed without data.")
                    return
        
        logger.info(f"Starting analysis of {len(self.df)} results...")
        logger.info(f"Models: {self.df['model'].nunique()}, Datasets: {self.df['dataset'].nunique()}")
        
        # 1. Performance Summary
        self._generate_performance_summary()
        
        # 2. Leaderboard
        self._generate_leaderboard()
        
        # 3. Visualizations
        self._generate_visualizations()
        
        # 4. Statistical Tests
        self._run_statistical_tests()
        
        # 5. Extended Analyses
        if run_all:
            self._run_extended_analyses()
        
        # 6. Generate final report
        self._generate_final_report()
        
        logger.info(f"Analysis complete. Reports saved to: {self.report_dir}")
    
    def _generate_performance_summary(self):
        """Generate comprehensive performance summary."""
        logger.info("Generating performance summary...")
        
        # Metrics to summarize
        metrics = ['R2', 'Q2', 'RMSE', 'MAE', 'Spearman', 'Pearson', 'CCC', 
                   'rm2_avg', 'within_1.0_log']
        available_metrics = [m for m in metrics if m in self.df.columns]
        
        # Summary by model
        summary = self.df.groupby('model')[available_metrics].agg(['mean', 'std', 'min', 'max',                                                                    'median', 'count'])
        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.sort_values('R2_mean', ascending=False)
        summary.to_csv(self.report_dir / "model_performance_summary.csv")
        
        # Summary by representation
        if 'representation' in self.df.columns:
            rep_summary = self.df.groupby('representation')[available_metrics].agg(['mean', 'std', 'count'])
            rep_summary.columns = ['_'.join(col).strip() for col in rep_summary.columns.values]
            rep_summary.to_csv(self.report_dir / "representation_performance_summary.csv")
        
        # Summary by dataset
        dataset_summary = self.df.groupby('dataset')[available_metrics].agg(['mean', 'std', 'min', 'max'])
        dataset_summary.columns = ['_'.join(col).strip() for col in dataset_summary.columns.values]
        dataset_summary.to_csv(self.report_dir / "dataset_performance_summary.csv")
        
        logger.info("Performance summary generated.")
    
    def _generate_leaderboard(self):
        """Generate various leaderboards."""
        logger.info("Generating leaderboards...")
        
        # Best model per dataset
        idx = self.df.groupby('dataset')['R2'].idxmax()
        leaderboard = self.df.loc[idx].sort_values('R2', ascending=False)
        
        cols = ['dataset', 'model', 'representation', 'R2', 'RMSE', 'MAE', 'Q2', 'CCC']
        available_cols = [c for c in cols if c in leaderboard.columns]
        leaderboard[available_cols].to_csv(self.report_dir / "leaderboard_best_per_dataset.csv", index=False)
        
        # Model win counts
        win_counts = self.df.loc[idx, 'model'].value_counts()
        win_counts.to_frame('wins').to_csv(self.report_dir / "model_win_counts.csv")
        
        # Overall model ranking
        model_ranking = self.df.groupby('model').agg({
            'R2': ['mean', 'std', 'count'],
            'RMSE': 'mean' if 'RMSE' in self.df.columns else 'count'
        })
        model_ranking.columns = ['R2_mean', 'R2_std', 'n_datasets', 'RMSE_mean']
        model_ranking['wins'] = win_counts.reindex(model_ranking.index).fillna(0).astype(int)
        model_ranking = model_ranking.sort_values('R2_mean', ascending=False)
        model_ranking['rank'] = range(1, len(model_ranking) + 1)
        model_ranking.to_csv(self.report_dir / "model_overall_ranking.csv")
        
        # Best model-representation combinations
        combo_ranking = self.df.groupby(['model', 'representation'])['R2'].agg(['mean', 'std', 'count'])
        combo_ranking = combo_ranking.sort_values('mean', ascending=False)
        combo_ranking.to_csv(self.report_dir / "model_representation_ranking.csv")
        
        logger.info("Leaderboards generated.")
    
    def _generate_visualizations(self):
        """Generate all visualizations."""
        logger.info("Generating visualizations...")
        
        # Heatmaps for different metrics
        for metric in ['R2', 'RMSE', 'Q2']:
            if metric in self.df.columns:
                plot_heatmap(self.df, self.report_dir, metric=metric)
        
        # Metric comparison
        plot_metric_comparison(self.df, self.report_dir)
        
        # Representation analysis
        plot_representation_analysis(self.df, self.report_dir)
        
        # Model performance distribution
        self._plot_model_distributions()
        
        logger.info("Visualizations generated.")
    
    def _plot_model_distributions(self):
        """Plot R² distribution for each model."""
        models = self.df['model'].unique()
        n_models = len(models)
        
        n_cols = min(4, n_models)
        n_rows = (n_models + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
        axes = np.atleast_2d(axes).flatten()
        
        for idx, model in enumerate(sorted(models)):
            ax = axes[idx]
            model_data = self.df[self.df['model'] == model]['R2']
            
            ax.hist(model_data, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            ax.axvline(x=model_data.mean(), color='red', linestyle='--', lw=2, 
                      label=f'Mean={model_data.mean():.3f}')
            ax.axvline(x=0.6, color='orange', linestyle=':', lw=1.5)
            ax.set_title(model[:25], fontsize=9)
            ax.set_xlabel('R²')
            ax.set_ylabel('Count')
            ax.legend(fontsize=7)
        
        # Hide empty axes
        for idx in range(n_models, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.savefig(self.report_dir / "model_r2_distributions.png", dpi=300)
        plt.close()
    
    def _run_statistical_tests(self):
        """Run statistical comparison tests."""
        logger.info("Running statistical tests...")
        
        # Friedman test
        stat, p_val, pivot_clean = perform_friedman_test(self.df, self.report_dir)
        
        if pivot_clean is not None:
            # Critical difference plot
            plot_critical_difference(pivot_clean, self.report_dir)
            
            # Nemenyi post-hoc test if Friedman is significant
            if p_val is not None and p_val < 0.05:
                perform_posthoc_nemenyi(pivot_clean, self.report_dir)
        
        # Pairwise Wilcoxon tests
        perform_pairwise_wilcoxon(self.df, self.report_dir)
        
        # Effect sizes
        calculate_effect_sizes(self.df, self.report_dir)
        
        # Kruskal-Wallis test for representations
        if 'representation' in self.df.columns and self.df['representation'].nunique() > 1:
            self._test_representation_effect()
        
        logger.info("Statistical tests completed.")
    
    def _test_representation_effect(self):
        """Test if representation has significant effect on performance."""
        groups = [group['R2'].values for name, group in self.df.groupby('representation')]
        
        if len(groups) >= 2 and all(len(g) >= 5 for g in groups):
            stat, p_val = kruskal(*groups)
            
            with open(self.report_dir / "representation_effect_test.txt", "w") as f:
                f.write("=" * 50 + "\n")
                f.write("REPRESENTATION EFFECT TEST (Kruskal-Wallis)\n")
                f.write("=" * 50 + "\n\n")
                f.write(f"H-statistic: {stat:.4f}\n")
                f.write(f"P-value: {p_val:.6e}\n")
                f.write(f"Significant at a=0.05: {p_val < 0.05}\n\n")
                
                f.write("Representation Performance Summary:\n")
                for rep in self.df['representation'].unique():
                    rep_data = self.df[self.df['representation'] == rep]['R2']
                    f.write(f"  {rep}: mean={rep_data.mean():.3f}, std={rep_data.std():.3f}, n={len(rep_data)}\n")
    
    def _run_extended_analyses(self):
        """Run extended QSAR-specific analyses."""
        logger.info("Running extended analyses...")
        
        # Error analysis by activity class
        try:
            analyze_errors_by_activity_class(self.output_dir, self.df)
        except Exception as e:
            logger.warning(f"Error analysis by activity class failed: {e}")
        
        # Dataset difficulty analysis
        try:
            difficulty_df = analyze_dataset_difficulty(self.df, self.report_dir)
        except Exception as e:
            logger.warning(f"Dataset difficulty analysis failed: {e}")
        
        # Model stability analysis
        try:
            stability_df = analyze_model_stability(self.df, self.report_dir, n_bootstrap=500)
        except Exception as e:
            logger.warning(f"Model stability analysis failed: {e}")
        
        # Ranking stability analysis
        try:
            ranking_df = analyze_ranking_stability(self.df, self.report_dir, n_bootstrap=500)
        except Exception as e:
            logger.warning(f"Ranking stability analysis failed: {e}")
        
        # QSAR validation report
        try:
            generate_qsar_validation_report(self.df, self.report_dir)
        except Exception as e:
            logger.warning(f"QSAR validation report failed: {e}")
        
        # Generate diagnostic plots for top models
        self._generate_diagnostic_plots_top_models(n_top=3)
        
        logger.info("Extended analyses completed.")
    
    def _generate_diagnostic_plots_top_models(self, n_top: int = 3):
        """Generate diagnostic plots for top performing models."""
        logger.info(f"Generating diagnostic plots for top {n_top} models...")
        
        # Get top models by mean R²
        top_models = self.df.groupby('model')['R2'].mean().nlargest(n_top).index.tolist()
        
        for model in top_models:
            model_files = self.df[self.df['model'] == model]['source_file'].tolist()
            
            if not model_files:
                continue
            
            # Aggregate predictions across all datasets for this model
            all_y_true = []
            all_y_pred = []
            
            for fname in model_files[:20]:  # Limit to 20 files
                file_path = self.output_dir / fname
                if file_path.exists():
                    data = load_predictions_from_file(file_path)
                    if data is not None:
                        all_y_true.extend(data['pIC50'].values)
                        all_y_pred.extend(data['pred'].values)
            
            if len(all_y_true) > 10:
                y_true = np.array(all_y_true)
                y_pred = np.array(all_y_pred)
                
                # Generate diagnostic plot
                safe_name = model.replace('/', '_').replace(' ', '_').replace('__', '_')
                plot_residual_diagnostics(y_true, y_pred, safe_name, self.report_dir)
    
    def _generate_final_report(self):
        """Generate comprehensive final report."""
        logger.info("Generating final report...")
        
        report_lines = []
        report_lines.append("=" * 70)
        report_lines.append("QSAR BENCHMARK ANALYSIS - FINAL REPORT")
        report_lines.append("=" * 70)
        report_lines.append("")
        report_lines.append(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"Output directory: {self.output_dir}")
        report_lines.append("")
        
        # Overview
        report_lines.append("-" * 50)
        report_lines.append("OVERVIEW")
        report_lines.append("-" * 50)
        report_lines.append(f"Total prediction files analyzed: {len(self.df)}")
        report_lines.append(f"Number of unique models: {self.df['model'].nunique()}")
        report_lines.append(f"Number of unique datasets: {self.df['dataset'].nunique()}")
        report_lines.append(f"Number of unique representations: {self.df['representation'].nunique()}")
        report_lines.append("")
        
        # Performance Summary
        report_lines.append("-" * 50)
        report_lines.append("PERFORMANCE SUMMARY")
        report_lines.append("-" * 50)
        
        model_perf = self.df.groupby('model')['R2'].agg(['mean', 'std', 'count'])
        model_perf = model_perf.sort_values('mean', ascending=False)
        
        report_lines.append("\nTop 10 Models by Mean R²:")
        for i, (model, row) in enumerate(model_perf.head(10).iterrows(), 1):
            report_lines.append(f"  {i:2d}. {model[:40]:<40} R²={row['mean']:.4f} (±{row['std']:.4f}), n={int(row['count'])}")
        
        # Dataset coverage
        report_lines.append("\n" + "-" * 50)
        report_lines.append("DATASET COVERAGE")
        report_lines.append("-" * 50)
        
        dataset_counts = self.df.groupby('model')['dataset'].nunique().sort_values(ascending=False)
        total_datasets = self.df['dataset'].nunique()
        
        report_lines.append(f"\nModels with complete coverage ({total_datasets} datasets):")
        complete_models = dataset_counts[dataset_counts == total_datasets].index.tolist()
        for model in complete_models[:10]:
            report_lines.append(f"  - {model}")
        
        if len(complete_models) > 10:
            report_lines.append(f"  ... and {len(complete_models) - 10} more")
        
        # Quality thresholds
        report_lines.append("\n" + "-" * 50)
        report_lines.append("QUALITY THRESHOLDS")
        report_lines.append("-" * 50)
        
        total = len(self.df)
        report_lines.append(f"\nPercentage of results meeting quality thresholds:")
        report_lines.append(f"  R² > 0.7: {(self.df['R2'] > 0.7).sum() / total * 100:.1f}%")
        report_lines.append(f"  R² > 0.6: {(self.df['R2'] > 0.6).sum() / total * 100:.1f}%")
        report_lines.append(f"  R² > 0.5: {(self.df['R2'] > 0.5).sum() / total * 100:.1f}%")
        report_lines.append(f"  R² < 0.3: {(self.df['R2'] < 0.3).sum() / total * 100:.1f}%")
        
        # Best combinations
        report_lines.append("\n" + "-" * 50)
        report_lines.append("BEST MODEL-REPRESENTATION COMBINATIONS")
        report_lines.append("-" * 50)
        
        combo_perf = self.df.groupby(['model', 'representation'])['R2'].mean().sort_values(ascending=False)
        report_lines.append("\nTop 10 combinations:")
        for i, ((model, rep), r2) in enumerate(combo_perf.head(10).items(), 1):
            report_lines.append(f"  {i:2d}. {model[:30]:<30} + {rep:<15} R²={r2:.4f}")
        
        # Win distribution
        report_lines.append("\n" + "-" * 50)
        report_lines.append("MODEL WINS DISTRIBUTION")
        report_lines.append("-" * 50)
        
        idx = self.df.groupby('dataset')['R2'].idxmax()
        win_counts = self.df.loc[idx, 'model'].value_counts()
        
        report_lines.append("\nNumber of datasets where each model achieved best R²:")
        for model, wins in win_counts.head(10).items():
            pct = wins / len(idx) * 100
            report_lines.append(f"  {model[:40]:<40} {wins:3d} wins ({pct:.1f}%)")
        
        # Files generated
        report_lines.append("\n" + "-" * 50)
        report_lines.append("FILES GENERATED")
        report_lines.append("-" * 50)
        
        report_files = list(self.report_dir.glob("*"))
        report_lines.append(f"\nTotal files in {self.report_dir}:")
        for f in sorted(report_files):
            report_lines.append(f"  - {f.name}")
        
        # Save report
        report_text = "\n".join(report_lines)
        with open(self.report_dir / "FINAL_REPORT.txt", "w") as f:
            f.write(report_text)
        
        # Also print summary to console
        print("\n" + "=" * 70)
        print("ANALYSIS COMPLETE - KEY FINDINGS")
        print("=" * 70)
        print(f"\nBest overall model: {model_perf.index[0]}")
        print(f"  Mean R² = {model_perf.iloc[0]['mean']:.4f} (±{model_perf.iloc[0]['std']:.4f})")
        print(f"\nModel with most wins: {win_counts.index[0]} ({win_counts.iloc[0]} datasets)")
        print(f"\nResults above R²>0.6 threshold: {(self.df['R2'] > 0.6).sum() / total * 100:.1f}%")
        print(f"\nReports saved to: {self.report_dir}")
        print("=" * 70)
        
        logger.info("Final report generated.")


# ============================================================================== #
# 10. Main Entry Point
# ============================================================================== #

def main():
    """Main entry point for the QSAR analyzer."""
    
    # Handle Jupyter notebook environment
    if 'ipykernel' in sys.modules:
        sys.argv = ['']
    
    parser = argparse.ArgumentParser(
        description="Comprehensive QSAR Benchmark Results Analyzer",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  python qsar_analyzer.py --dir ./results
  python qsar_analyzer.py --dir ./results --rebuild
  python qsar_analyzer.py --dir ./results --quick
        """
    )
    
    parser.add_argument(
        "--dir", 
        type=str, 
        default="./qsar_compact_out",
        help="Directory containing result CSVs (default: ./qsar_compact_out)"
    )
    parser.add_argument(
        "--rebuild", 
        action="store_true",
        help="Force rebuild of summary CSV even if it exists"
    )
    parser.add_argument(
        "--quick", 
        action="store_true",
        help="Run quick analysis (skip bootstrap and extended analyses)"
    )
    parser.add_argument(
        "--bootstrap", 
        type=int, 
        default=500,
        help="Number of bootstrap iterations (default: 500)"
    )
    
    args = parser.parse_args()
    
    target_dir = Path(args.dir)
    
    if not target_dir.exists():
        logger.error(f"Directory not found: {target_dir}")
        print(f"\nError: Directory '{target_dir}' does not exist.")
        print("Please provide a valid directory containing QSAR prediction files.")
        sys.exit(1)
    
    # Initialize analyzer
    analyzer = QSARAnalyzer(target_dir)
    
    # Build or load summary
    if args.rebuild or not analyzer.summary_path.exists():
        success = analyzer.build_summary()
        if not success:
            logger.error("Failed to build summary. Exiting.")
            sys.exit(1)
    else:
        analyzer.load_summary()
    
    # Run analysis
    analyzer.analyze(run_all=not args.quick)
    
    print(f"\n✓ Analysis complete! Check {analyzer.report_dir} for results.")


if __name__ == "__main__":
    main()